# CReST: V5/V6 Model Development

Documents development-stage V5/V6 experiments used to diagnose assessment-design leakage. This notebook is not part of the locked confirmatory training pipeline.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Standalone CReST V5 modelling environment

from pathlib import Path
import json
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler,
    Subset
)

import matplotlib.pyplot as plt


# Paths

BASE_DIR = Path.cwd()

HANDOFF_DIR = (
    BASE_DIR
    / "processed"
    / "crest_v5_handoff"
)

FIG_DIR = (
    BASE_DIR
    / "figures"
)

CHECKPOINT_DIR = (
    BASE_DIR
    / "checkpoints"
)

RESULTS_DIR = (
    BASE_DIR
    / "results"
)


FIG_DIR.mkdir(
    exist_ok=True
)

CHECKPOINT_DIR.mkdir(
    exist_ok=True
)

RESULTS_DIR.mkdir(
    exist_ok=True
)


# Verify handoff directory

if not HANDOFF_DIR.exists():

    raise FileNotFoundError(
        f"Handoff directory not found:\n{HANDOFF_DIR}\n\n"
        "Check that the notebook is located in "
        "'Dataset and codes'."
    )


print(
    "Working directory:",
    BASE_DIR
)

print(
    "Handoff directory:",
    HANDOFF_DIR
)


# Load configuration

with open(
    HANDOFF_DIR / "handoff_config.json",
    "r",
    encoding="utf-8"
) as f:

    CONFIG = json.load(f)


GLOBAL_SEED = int(
    CONFIG["global_seed"]
)

DOMAIN_ORDER = (
    CONFIG["domain_order"]
)

PROCESS_ORDER = (
    CONFIG["process_order"]
)


# Reproducibility

def set_global_seed(
    seed=GLOBAL_SEED
):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_global_seed()


# Device

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print(
    "\nCReST version:",
    CONFIG["version"]
)

print(
    "Students:",
    CONFIG["n_students"]
)

print(
    "Items:",
    CONFIG["n_items"]
)

print(
    "Seed:",
    GLOBAL_SEED
)

print(
    "Device:",
    device
)

Working directory: PROJECT_ROOT
Handoff directory: PROJECT_ROOT\processed\crest_v5_handoff

CReST version: CReST-V5
Students: 142564
Items: 32
Seed: 20261001
Device: cpu


In [ ]:
# Load only the data required for CReST V5 modelling

# Core arrays - memory mapped

credit_tokens = np.load(
    HANDOFF_DIR / "credit_tokens.npy",
    mmap_mode="r"
)

behaviour_tokens = np.load(
    HANDOFF_DIR / "behaviour_tokens.npy",
    mmap_mode="r"
)

performance_mask = np.load(
    HANDOFF_DIR / "performance_mask.npy",
    mmap_mode="r"
)

interaction_mask = np.load(
    HANDOFF_DIR / "interaction_mask.npy",
    mmap_mode="r"
)

student_design_mask = np.load(
    HANDOFF_DIR / "student_design_mask.npy",
    mmap_mode="r"
)

process_tensor = np.load(
    HANDOFF_DIR / "process_tensor.npy",
    mmap_mode="r"
)

booklet_nuisance_ids = np.load(
    HANDOFF_DIR / "booklet_nuisance_ids.npy",
    mmap_mode="r"
)

item_domain_ids = np.load(
    HANDOFF_DIR / "item_domain_ids.npy",
    mmap_mode="r"
)

item_ideation_ids = np.load(
    HANDOFF_DIR / "item_ideation_ids.npy",
    mmap_mode="r"
)


# DataFrames required for modelling

model_context = pd.read_pickle(
    HANDOFF_DIR / "model_context.pkl"
)

primary_meta = pd.read_pickle(
    HANDOFF_DIR / "primary_meta.pkl"
)

item_registry = pd.read_csv(
    HANDOFF_DIR / "item_registry.csv"
)

context_registry_locked = pd.read_csv(
    HANDOFF_DIR / "context_registry_locked.csv"
)

country_fold_df = pd.read_csv(
    HANDOFF_DIR / "country_fold_assignment_7fold.csv"
)

country_split_protocol_df = pd.read_csv(
    HANDOFF_DIR / "country_split_protocol_7fold.csv"
)


# Lightweight integrity checks

N_STUDENTS = int(
    CONFIG["n_students"]
)

N_ITEMS = int(
    CONFIG["n_items"]
)


assert credit_tokens.shape == (
    N_STUDENTS,
    N_ITEMS
)

assert behaviour_tokens.shape == (
    N_STUDENTS,
    N_ITEMS
)

assert process_tensor.shape == (
    N_STUDENTS,
    N_ITEMS,
    3
)

assert len(
    primary_meta
) == N_STUDENTS

assert len(
    model_context
) == N_STUDENTS

assert (
    primary_meta["CNTSTUID"].nunique()
    == N_STUDENTS
)

assert (
    primary_meta["CNT"].nunique()
    == 63
)

assert sorted(
    primary_meta["country_fold"]
    .unique()
    .tolist()
) == [
    0, 1, 2, 3, 4, 5, 6
]


# Summary

print("credit_tokens:", credit_tokens.shape)
print("behaviour_tokens:", behaviour_tokens.shape)
print("process_tensor:", process_tensor.shape)

print(
    "model_context:",
    model_context.shape
)

print(
    "primary_meta:",
    primary_meta.shape
)

print(
    "Education systems:",
    primary_meta["CNT"].nunique()
)

print(
    "Country folds:",
    sorted(
        primary_meta["country_fold"]
        .unique()
        .tolist()
    )
)

print(
    "Booklet nuisance classes:",
    len(
        np.unique(
            booklet_nuisance_ids
        )
    )
)

print(
    "\nCore CReST V5 modelling data loaded successfully."
)

credit_tokens: (142564, 32)
behaviour_tokens: (142564, 32)
process_tensor: (142564, 32, 3)
model_context: (142564, 41)
primary_meta: (142564, 6)
Education systems: 63
Country folds: [0, 1, 2, 3, 4, 5, 6]
Booklet nuisance classes: 30

Core CReST V5 modelling data loaded successfully.


In [ ]:
# Restore modelling variables and Fold-0 split

# Context variable sets

common_student_vars = (
    CONFIG["common_student_vars"]
)

common_school_vars = (
    CONFIG["common_school_vars"]
)

context_common_vars = (
    CONFIG["context_common_vars"]
)


print(
    "Student context:",
    common_student_vars
)

print(
    "School context:",
    common_school_vars
)


# Fold helper

N_COUNTRY_FOLDS = int(
    CONFIG["n_country_folds"]
)


def get_outer_split_indices(
    outer_fold
):

    test_fold = outer_fold

    validation_fold = (
        outer_fold + 1
    ) % N_COUNTRY_FOLDS

    fold_values = (
        primary_meta[
            "country_fold"
        ]
        .to_numpy()
    )

    train_idx = np.where(
        ~np.isin(
            fold_values,
            [
                test_fold,
                validation_fold
            ]
        )
    )[0]

    validation_idx = np.where(
        fold_values
        == validation_fold
    )[0]

    test_idx = np.where(
        fold_values
        == test_fold
    )[0]


    return {
        "train_idx": train_idx,
        "validation_idx": validation_idx,
        "test_idx": test_idx,
        "validation_fold": validation_fold,
        "test_fold": test_fold
    }


# Fold 0

split0 = get_outer_split_indices(
    0
)


print(
    "\nFold 0"
)

print(
    "Train:",
    len(
        split0["train_idx"]
    )
)

print(
    "Validation:",
    len(
        split0["validation_idx"]
    )
)

print(
    "Test:",
    len(
        split0["test_idx"]
    )
)


print(
    "\nTrain systems:",
    primary_meta.iloc[
        split0["train_idx"]
    ]["CNT"].nunique()
)

print(
    "Validation systems:",
    primary_meta.iloc[
        split0["validation_idx"]
    ]["CNT"].nunique()
)

print(
    "Test systems:",
    primary_meta.iloc[
        split0["test_idx"]
    ]["CNT"].nunique()
)

Student context: ['ST004D01T', 'AGE', 'REPEAT', 'HOMEPOS', 'IMMIG', 'HOME_TEST_LANG_MATCH']
School context: ['SC001Q01TA', 'SRESPCUR', 'SRESPRES']

Fold 0
Train: 99966
Validation: 20949
Test: 21649

Train systems: 45
Validation systems: 9
Test systems: 9


In [ ]:
# Leakage-safe preprocessing classes

class FoldContextPreprocessor:

    def __init__(
        self,
        registry,
        variables
    ):

        self.variables = list(
            variables
        )

        reg = registry.loc[
            registry["variable"].isin(
                self.variables
            )
        ].copy()

        self.continuous_vars = (
            reg.loc[
                reg["feature_type"] == "continuous",
                "variable"
            ]
            .tolist()
        )

        self.categorical_vars = (
            reg.loc[
                reg["feature_type"].isin(
                    [
                        "categorical",
                        "binary",
                        "ordinal"
                    ]
                ),
                "variable"
            ]
            .tolist()
        )

        self.medians = {}
        self.iqrs = {}
        self.category_maps = {}


    def fit(
        self,
        df,
        train_idx
    ):

        train_df = df.iloc[
            train_idx
        ]


        # Continuous
        for var in self.continuous_vars:

            x = pd.to_numeric(
                train_df[var],
                errors="coerce"
            )

            median = x.median()

            q25 = x.quantile(
                0.25
            )

            q75 = x.quantile(
                0.75
            )

            iqr = q75 - q25


            if (
                pd.isna(iqr)
                or
                iqr < 1e-8
            ):
                iqr = 1.0


            self.medians[var] = float(
                median
            )

            self.iqrs[var] = float(
                iqr
            )


        # Categorical
        for var in self.categorical_vars:

            values = (
                train_df[var]
                .dropna()
                .unique()
                .tolist()
            )

            values = sorted(
                values,
                key=lambda x: str(x)
            )


            # 0 = missing
            # 1 = unseen category
            # 2+ = observed category
            self.category_maps[var] = {
                value: i + 2
                for i, value
                in enumerate(values)
            }


        return self


    def transform(
        self,
        df
    ):

        # Continuous

        continuous_values = []
        continuous_masks = []


        for var in self.continuous_vars:

            x = pd.to_numeric(
                df[var],
                errors="coerce"
            ).to_numpy(
                dtype=np.float32
            )

            mask = ~np.isnan(
                x
            )

            scaled = (
                x
                -
                self.medians[var]
            ) / self.iqrs[var]

            scaled[
                ~mask
            ] = 0.0


            continuous_values.append(
                scaled
            )

            continuous_masks.append(
                mask
            )


        if continuous_values:

            continuous = np.column_stack(
                continuous_values
            ).astype(
                np.float32
            )

            continuous_mask = np.column_stack(
                continuous_masks
            )

        else:

            continuous = np.empty(
                (
                    len(df),
                    0
                ),
                dtype=np.float32
            )

            continuous_mask = np.empty(
                (
                    len(df),
                    0
                ),
                dtype=bool
            )


        # Categorical

        categorical_values = []


        for var in self.categorical_vars:

            mapping = (
                self.category_maps[var]
            )

            values = df[var].to_numpy()

            encoded = np.zeros(
                len(df),
                dtype=np.int64
            )


            for i, value in enumerate(
                values
            ):

                if pd.isna(
                    value
                ):
                    encoded[i] = 0

                else:
                    encoded[i] = (
                        mapping.get(
                            value,
                            1
                        )
                    )


            categorical_values.append(
                encoded
            )


        if categorical_values:

            categorical = np.column_stack(
                categorical_values
            )

        else:

            categorical = np.empty(
                (
                    len(df),
                    0
                ),
                dtype=np.int64
            )


        return {
            "continuous":
                continuous,

            "continuous_mask":
                continuous_mask,

            "categorical":
                categorical
        }


    def categorical_cardinalities(
        self
    ):

        return [
            len(
                self.category_maps[var]
            ) + 2

            for var
            in self.categorical_vars
        ]


class ProcessRobustPreprocessor:

    def fit(
        self,
        x_train
    ):

        self.median = np.nanmedian(
            x_train,
            axis=0
        )

        q25 = np.nanpercentile(
            x_train,
            25,
            axis=0
        )

        q75 = np.nanpercentile(
            x_train,
            75,
            axis=0
        )


        self.iqr = (
            q75 - q25
        )


        self.median[
            ~np.isfinite(
                self.median
            )
        ] = 0.0


        self.iqr[
            ~np.isfinite(
                self.iqr
            )
        ] = 1.0


        self.iqr[
            self.iqr < 1e-8
        ] = 1.0


        return self


    def transform(
        self,
        x
    ):

        mask = ~np.isnan(
            x
        )

        scaled = (
            x
            -
            self.median[
                None, :, :
            ]
        ) / self.iqr[
            None, :, :
        ]


        scaled = np.asarray(
            scaled,
            dtype=np.float32
        )


        scaled[
            ~mask
        ] = 0.0


        return (
            scaled,
            mask
        )


print(
    "Preprocessing classes ready."
)

Preprocessing classes ready.


In [ ]:
# Build leakage-safe Fold-0 modelling bundle

def build_fold_bundle(
    outer_fold
):

    split = get_outer_split_indices(
        outer_fold
    )

    train_idx = (
        split["train_idx"]
    )

    validation_idx = (
        split["validation_idx"]
    )

    test_idx = (
        split["test_idx"]
    )


    # Context preprocessors

    student_pp = (
        FoldContextPreprocessor(
            context_registry_locked,
            common_student_vars
        )
        .fit(
            model_context,
            train_idx
        )
    )


    school_pp = (
        FoldContextPreprocessor(
            context_registry_locked,
            common_school_vars
        )
        .fit(
            model_context,
            train_idx
        )
    )


    # Process preprocessor

    process_pp = (
        ProcessRobustPreprocessor()
        .fit(
            process_tensor[
                train_idx
            ]
        )
    )


    # Training-country labels

    train_countries = sorted(
        primary_meta.iloc[
            train_idx
        ]["CNT"]
        .unique()
        .tolist()
    )


    country_to_train_id = {
        cnt: i

        for i, cnt
        in enumerate(
            train_countries
        )
    }


    # Build one data split

    def make_split(
        indices
    ):

        student_ctx = (
            student_pp.transform(
                model_context.iloc[
                    indices
                ]
            )
        )

        school_ctx = (
            school_pp.transform(
                model_context.iloc[
                    indices
                ]
            )
        )


        proc_values, proc_mask = (
            process_pp.transform(
                process_tensor[
                    indices
                ]
            )
        )


        countries = (
            primary_meta.iloc[
                indices
            ]["CNT"]
            .to_numpy()
        )


        country_ids = np.array(
            [
                country_to_train_id.get(
                    cnt,
                    -1
                )

                for cnt
                in countries
            ],
            dtype=np.int64
        )


        return {

            "indices":
                indices,

            "credit_tokens":
                np.asarray(
                    credit_tokens[
                        indices
                    ],
                    dtype=np.int64
                ),

            "behaviour_tokens":
                np.asarray(
                    behaviour_tokens[
                        indices
                    ],
                    dtype=np.int64
                ),

            "performance_mask":
                np.asarray(
                    performance_mask[
                        indices
                    ],
                    dtype=bool
                ),

            "interaction_mask":
                np.asarray(
                    interaction_mask[
                        indices
                    ],
                    dtype=bool
                ),

            "design_mask":
                np.asarray(
                    student_design_mask[
                        indices
                    ],
                    dtype=bool
                ),

            "process_values":
                proc_values,

            "process_mask":
                proc_mask,

            "student_continuous":
                student_ctx[
                    "continuous"
                ],

            "student_continuous_mask":
                student_ctx[
                    "continuous_mask"
                ],

            "student_categorical":
                student_ctx[
                    "categorical"
                ],

            "school_continuous":
                school_ctx[
                    "continuous"
                ],

            "school_continuous_mask":
                school_ctx[
                    "continuous_mask"
                ],

            "school_categorical":
                school_ctx[
                    "categorical"
                ],

            "booklet_nuisance_id":
                np.asarray(
                    booklet_nuisance_ids[
                        indices
                    ],
                    dtype=np.int64
                ),

            "country_train_id":
                country_ids,

            "CNT":
                countries,

            "CNTSCHID":
                primary_meta.iloc[
                    indices
                ][
                    "CNTSCHID"
                ].to_numpy(),

            "CNTSTUID":
                primary_meta.iloc[
                    indices
                ][
                    "CNTSTUID"
                ].to_numpy()
        }


    return {

        "outer_fold":
            outer_fold,

        "train":
            make_split(
                train_idx
            ),

        "validation":
            make_split(
                validation_idx
            ),

        "test":
            make_split(
                test_idx
            ),

        "student_preprocessor":
            student_pp,

        "school_preprocessor":
            school_pp,

        "process_preprocessor":
            process_pp,

        "country_to_train_id":
            country_to_train_id
    }


fold0 = build_fold_bundle(
    0
)


print(
    "Train:",
    len(
        fold0["train"]["CNT"]
    )
)

print(
    "Validation:",
    len(
        fold0["validation"]["CNT"]
    )
)

print(
    "Test:",
    len(
        fold0["test"]["CNT"]
    )
)

print(
    "Training country classes:",
    len(
        fold0["country_to_train_id"]
    )
)

print(
    "Student continuous:",
    fold0[
        "train"
    ][
        "student_continuous"
    ].shape
)

print(
    "Student categorical:",
    fold0[
        "train"
    ][
        "student_categorical"
    ].shape
)

print(
    "School categorical:",
    fold0[
        "train"
    ][
        "school_categorical"
    ].shape
)

print(
    "Process:",
    fold0[
        "train"
    ][
        "process_values"
    ].shape
)

Train: 99966
Validation: 20949
Test: 21649
Training country classes: 45
Student continuous: (99966, 2)
Student categorical: (99966, 4)
School categorical: (99966, 3)
Process: (99966, 32, 3)


In [ ]:
# PyTorch datasets and country-balanced loaders

class CReSTDataset(Dataset):

    def __init__(
        self,
        split_data
    ):
        self.d = split_data


    def __len__(
        self
    ):
        return len(
            self.d["CNTSTUID"]
        )


    def __getitem__(
        self,
        idx
    ):

        return {

            "credit_tokens":
                torch.as_tensor(
                    self.d["credit_tokens"][idx],
                    dtype=torch.long
                ),

            "behaviour_tokens":
                torch.as_tensor(
                    self.d["behaviour_tokens"][idx],
                    dtype=torch.long
                ),

            "performance_mask":
                torch.as_tensor(
                    self.d["performance_mask"][idx],
                    dtype=torch.bool
                ),

            "interaction_mask":
                torch.as_tensor(
                    self.d["interaction_mask"][idx],
                    dtype=torch.bool
                ),

            "design_mask":
                torch.as_tensor(
                    self.d["design_mask"][idx],
                    dtype=torch.bool
                ),

            "process_values":
                torch.as_tensor(
                    self.d["process_values"][idx],
                    dtype=torch.float32
                ),

            "process_mask":
                torch.as_tensor(
                    self.d["process_mask"][idx],
                    dtype=torch.bool
                ),

            "student_continuous":
                torch.as_tensor(
                    self.d["student_continuous"][idx],
                    dtype=torch.float32
                ),

            "student_continuous_mask":
                torch.as_tensor(
                    self.d["student_continuous_mask"][idx],
                    dtype=torch.bool
                ),

            "student_categorical":
                torch.as_tensor(
                    self.d["student_categorical"][idx],
                    dtype=torch.long
                ),

            "school_continuous":
                torch.as_tensor(
                    self.d["school_continuous"][idx],
                    dtype=torch.float32
                ),

            "school_continuous_mask":
                torch.as_tensor(
                    self.d["school_continuous_mask"][idx],
                    dtype=torch.bool
                ),

            "school_categorical":
                torch.as_tensor(
                    self.d["school_categorical"][idx],
                    dtype=torch.long
                ),

            "booklet_nuisance_id":
                torch.as_tensor(
                    self.d["booklet_nuisance_id"][idx],
                    dtype=torch.long
                ),

            "country_train_id":
                torch.as_tensor(
                    self.d["country_train_id"][idx],
                    dtype=torch.long
                )
        }


# Datasets

train_dataset = CReSTDataset(
    fold0["train"]
)

validation_dataset = CReSTDataset(
    fold0["validation"]
)

test_dataset = CReSTDataset(
    fold0["test"]
)


# Country-balanced training weights

train_countries = (
    fold0["train"]["CNT"]
)

unique_countries, country_counts = np.unique(
    train_countries,
    return_counts=True
)


country_count_map = {
    cnt: count
    for cnt, count
    in zip(
        unique_countries,
        country_counts
    )
}


sampling_weights = np.array(
    [
        1.0
        / country_count_map[cnt]

        for cnt
        in train_countries
    ],
    dtype=np.float64
)


sampler_generator = (
    torch.Generator()
    .manual_seed(
        GLOBAL_SEED
    )
)


train_sampler = WeightedRandomSampler(

    weights=torch.as_tensor(
        sampling_weights,
        dtype=torch.double
    ),

    num_samples=len(
        train_dataset
    ),

    replacement=True,

    generator=sampler_generator
)


# DataLoaders

BATCH_SIZE = int(
    CONFIG["batch_size"]
)


train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    sampler=train_sampler,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


validation_loader = DataLoader(

    validation_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(validation_loader)
)

print(
    "Test batches:",
    len(test_loader)
)

print(
    "Training systems:",
    len(unique_countries)
)

Batch size: 256
Train batches: 391
Validation batches: 82
Test batches: 85
Training systems: 45


In [ ]:
# DataLoader and sampler sanity check

# Inspect one training batch

batch = next(
    iter(
        train_loader
    )
)


print("Batch tensor shapes:")

for key, value in batch.items():

    print(
        f"{key:27s}",
        tuple(
            value.shape
        ),
        value.dtype
    )


# Quick country-balancing audit

audit_sampler = WeightedRandomSampler(

    weights=torch.as_tensor(
        sampling_weights,
        dtype=torch.double
    ),

    num_samples=20000,

    replacement=True,

    generator=torch.Generator()
    .manual_seed(
        GLOBAL_SEED + 1
    )
)


sampled_indices = np.array(
    list(
        iter(
            audit_sampler
        )
    )
)


sampled_countries = (
    train_countries[
        sampled_indices
    ]
)


sampling_audit = (
    pd.Series(
        sampled_countries
    )
    .value_counts(
        normalize=True
    )
    .mul(100)
)


print(
    "\nExpected per-country share:",
    round(
        100 / len(unique_countries),
        3
    ),
    "%"
)


print(
    "Observed min:",
    round(
        sampling_audit.min(),
        3
    ),
    "%"
)

print(
    "Observed max:",
    round(
        sampling_audit.max(),
        3
    ),
    "%"
)


# Critical label checks

assert (
    fold0["train"]["country_train_id"]
    >= 0
).all()

assert (
    fold0["validation"]["country_train_id"]
    == -1
).all()

assert (
    fold0["test"]["country_train_id"]
    == -1
).all()


print(
    "\nCountry-label isolation: PASS"
)

Batch tensor shapes:
credit_tokens               (256, 32) torch.int64
behaviour_tokens            (256, 32) torch.int64
performance_mask            (256, 32) torch.bool
interaction_mask            (256, 32) torch.bool
design_mask                 (256, 32) torch.bool
process_values              (256, 32, 3) torch.float32
process_mask                (256, 32, 3) torch.bool
student_continuous          (256, 2) torch.float32
student_continuous_mask     (256, 2) torch.bool
student_categorical         (256, 4) torch.int64
school_continuous           (256, 0) torch.float32
school_continuous_mask      (256, 0) torch.bool
school_categorical          (256, 3) torch.int64
booklet_nuisance_id         (256,) torch.int64
country_train_id            (256,) torch.int64

Expected per-country share: 2.222 %
Observed min: 2.02 %
Observed max: 2.445 %

Country-label isolation: PASS


In [ ]:
# Core utilities for standalone CReST V5

# Gradient reversal

class GradientReversalFunction(
    torch.autograd.Function
):

    @staticmethod
    def forward(
        ctx,
        x,
        lambda_grl
    ):
        ctx.lambda_grl = lambda_grl
        return x.view_as(x)


    @staticmethod
    def backward(
        ctx,
        grad_output
    ):
        return (
            -ctx.lambda_grl * grad_output,
            None
        )


def grad_reverse(
    x,
    lambda_grl=1.0
):
    return GradientReversalFunction.apply(
        x,
        lambda_grl
    )


# Masked mean pooling

def masked_mean(
    x,
    mask,
    dim=1,
    eps=1e-8
):

    w = (
        mask
        .unsqueeze(-1)
        .to(x.dtype)
    )

    numerator = (
        (x * w)
        .sum(dim=dim)
    )

    denominator = (
        w.sum(dim=dim)
        .clamp_min(eps)
    )

    return numerator / denominator


# Context encoder

class ContextEncoder(nn.Module):

    def __init__(
        self,
        n_continuous,
        categorical_cardinalities,
        d_model=128,
        cat_embedding_dim=16,
        dropout=0.1
    ):

        super().__init__()

        self.n_continuous = (
            n_continuous
        )

        self.categorical_embeddings = (
            nn.ModuleList(
                [
                    nn.Embedding(
                        cardinality,
                        cat_embedding_dim
                    )
                    for cardinality
                    in categorical_cardinalities
                ]
            )
        )

        input_dim = (
            2 * n_continuous
            +
            len(
                categorical_cardinalities
            )
            * cat_embedding_dim
        )

        self.encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                d_model
            ),

            nn.GELU(),

            nn.LayerNorm(
                d_model
            ),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                d_model,
                d_model
            ),

            nn.GELU(),

            nn.LayerNorm(
                d_model
            )
        )


    def forward(
        self,
        continuous,
        continuous_mask,
        categorical
    ):

        parts = []


        if self.n_continuous > 0:

            parts.append(
                continuous
            )

            parts.append(
                continuous_mask.float()
            )


        for j, embedding in enumerate(
            self.categorical_embeddings
        ):

            parts.append(
                embedding(
                    categorical[:, j]
                )
            )


        x = torch.cat(
            parts,
            dim=1
        )

        return self.encoder(
            x
        )


print(
    "CReST V5 utility layers ready."
)

CReST V5 utility layers ready.


In [ ]:
# Standalone CReST V5 architecture

class CReSTV5(nn.Module):

    def __init__(
        self,
        item_domain_ids,
        item_ideation_ids,
        student_n_continuous,
        student_cat_cardinalities,
        school_n_continuous,
        school_cat_cardinalities,
        n_train_countries=45,
        n_booklets=30,
        n_domains=4,
        n_processes=3,
        d_model=128,
        latent_dim=64,
        n_heads=8,
        n_layers=3,
        dropout=0.1
    ):

        super().__init__()

        self.n_items = 32
        self.n_domains = n_domains
        self.n_processes = n_processes
        self.d_model = d_model
        self.latent_dim = latent_dim


        # Fixed item metadata

        self.register_buffer(
            "item_domain_ids",
            torch.as_tensor(
                np.asarray(
                    item_domain_ids
                ),
                dtype=torch.long
            )
        )

        self.register_buffer(
            "item_ideation_ids",
            torch.as_tensor(
                np.asarray(
                    item_ideation_ids
                ),
                dtype=torch.long
            )
        )


        # Embeddings
        #
        # Exact item identity is decoder-only.

        self.item_embedding = nn.Embedding(
            self.n_items,
            d_model
        )

        self.domain_embedding = nn.Embedding(
            n_domains,
            d_model
        )

        self.ideation_embedding = nn.Embedding(
            n_processes,
            d_model
        )


        # Credit:
        # 0 = unavailable
        # 1 = no credit
        # 2 = partial
        # 3 = full
        # 4 = SSL mask
        self.credit_embedding = nn.Embedding(
            5,
            d_model
        )


        # Behaviour:
        # 0 = unavailable
        # 1 = scorable
        # 2 = no response
        # 3 = not reached
        self.behaviour_embedding = nn.Embedding(
            4,
            d_model
        )


        # Process projection
        #
        # 3 values + 3 availability indicators

        self.process_projection = nn.Sequential(

            nn.Linear(
                6,
                d_model
            ),

            nn.GELU(),

            nn.LayerNorm(
                d_model
            )
        )


        # Performance Transformer

        performance_layer = (
            nn.TransformerEncoderLayer(

                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=4 * d_model,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True
            )
        )

        self.performance_encoder = (
            nn.TransformerEncoder(
                performance_layer,
                num_layers=n_layers,
                norm=nn.LayerNorm(
                    d_model
                )
            )
        )


        # Process Transformer

        process_layer = (
            nn.TransformerEncoderLayer(

                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=4 * d_model,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True
            )
        )

        self.process_encoder = (
            nn.TransformerEncoder(
                process_layer,
                num_layers=n_layers,
                norm=nn.LayerNorm(
                    d_model
                )
            )
        )


        # Cross-view attention

        self.cross_attention = (
            nn.MultiheadAttention(
                embed_dim=d_model,
                num_heads=n_heads,
                dropout=dropout,
                batch_first=True
            )
        )

        self.cross_norm = nn.LayerNorm(
            d_model
        )


        # Process-balanced general pooling

        self.process_balance_norm = (
            nn.LayerNorm(
                d_model
            )
        )


        # Context encoders

        self.student_encoder = ContextEncoder(

            n_continuous=(
                student_n_continuous
            ),

            categorical_cardinalities=(
                student_cat_cardinalities
            ),

            d_model=d_model,
            dropout=dropout
        )


        self.school_encoder = ContextEncoder(

            n_continuous=(
                school_n_continuous
            ),

            categorical_cardinalities=(
                school_cat_cardinalities
            ),

            d_model=d_model,
            dropout=dropout
        )


        self.context_fusion = nn.Sequential(

            nn.Linear(
                2 * d_model,
                d_model
            ),

            nn.GELU(),

            nn.LayerNorm(
                d_model
            )
        )


        # Latent projections

        self.creative_projection = nn.Sequential(

            nn.Linear(
                d_model,
                latent_dim
            ),

            nn.LayerNorm(
                latent_dim
            )
        )


        self.context_projection = nn.Sequential(

            nn.Linear(
                d_model,
                latent_dim
            ),

            nn.LayerNorm(
                latent_dim
            )
        )


        # Domain-general / domain-specific decomposition

        self.domain_shared_projection = nn.Sequential(

            nn.Linear(
                d_model,
                latent_dim
            ),

            nn.GELU(),

            nn.LayerNorm(
                latent_dim
            )
        )


        self.domain_specific_projections = (
            nn.ModuleList(
                [
                    nn.Sequential(

                        nn.Linear(
                            d_model,
                            latent_dim
                        ),

                        nn.GELU(),

                        nn.LayerNorm(
                            latent_dim
                        )
                    )

                    for _ in range(
                        n_domains
                    )
                ]
            )
        )


        # Reconstruction heads

        self.credit_head = nn.Linear(
            d_model,
            3
        )

        self.process_head = nn.Linear(
            d_model,
            3
        )


        self.domain_latent_decoder = nn.Sequential(

            nn.Linear(
                latent_dim,
                d_model
            ),

            nn.GELU(),

            nn.LayerNorm(
                d_model
            )
        )


        self.decomposition_credit_head = (
            nn.Linear(
                d_model,
                3
            )
        )


        # Context reconstruction

        self.total_context_continuous = (
            student_n_continuous
            +
            school_n_continuous
        )


        self.context_continuous_head = (
            nn.Linear(
                latent_dim,
                self.total_context_continuous
            )
            if self.total_context_continuous > 0
            else None
        )


        all_context_cardinalities = (
            list(
                student_cat_cardinalities
            )
            +
            list(
                school_cat_cardinalities
            )
        )


        self.context_categorical_heads = (
            nn.ModuleList(
                [
                    nn.Linear(
                        latent_dim,
                        cardinality
                    )

                    for cardinality
                    in all_context_cardinalities
                ]
            )
        )


        # Nuisance routing

        self.booklet_classifier = nn.Sequential(

            nn.Linear(
                latent_dim,
                latent_dim
            ),

            nn.GELU(),

            nn.Linear(
                latent_dim,
                n_booklets
            )
        )


        self.context_country_classifier = nn.Sequential(

            nn.Linear(
                latent_dim,
                latent_dim
            ),

            nn.GELU(),

            nn.Linear(
                latent_dim,
                n_train_countries
            )
        )


        self.creative_country_classifier = nn.Sequential(

            nn.Linear(
                latent_dim,
                latent_dim
            ),

            nn.GELU(),

            nn.Linear(
                latent_dim,
                n_train_countries
            )
        )


    # Encoder metadata
    #
    # NO exact item identity.

    def encoder_metadata(
        self,
        batch_size
    ):

        meta = (

            self.domain_embedding(
                self.item_domain_ids
            )

            +

            self.ideation_embedding(
                self.item_ideation_ids
            )
        )

        return (
            meta
            .unsqueeze(0)
            .expand(
                batch_size,
                -1,
                -1
            )
        )


    # Decoder metadata
    #
    # Exact item identity allowed here.

    def decoder_metadata(
        self,
        batch_size,
        device
    ):

        item_ids = torch.arange(
            self.n_items,
            device=device
        )

        meta = (

            self.item_embedding(
                item_ids
            )

            +

            self.domain_embedding(
                self.item_domain_ids
            )

            +

            self.ideation_embedding(
                self.item_ideation_ids
            )
        )

        return (
            meta
            .unsqueeze(0)
            .expand(
                batch_size,
                -1,
                -1
            )
        )


    # Process-balanced general pooling

    def process_balanced_pool(
        self,
        item_tokens,
        interaction_mask
    ):

        pooled_list = []
        available_list = []


        for process_id in range(
            self.n_processes
        ):

            process_selector = (

                self.item_ideation_ids
                .unsqueeze(0)

                == process_id
            )


            valid_mask = (

                interaction_mask

                &

                process_selector
            )


            available = (
                valid_mask.any(
                    dim=1
                )
            )


            pooled = masked_mean(
                item_tokens,
                valid_mask,
                dim=1
            )


            pooled = torch.where(

                available.unsqueeze(-1),

                pooled,

                torch.zeros_like(
                    pooled
                )
            )


            pooled_list.append(
                pooled
            )

            available_list.append(
                available
            )


        process_pooled = torch.stack(
            pooled_list,
            dim=1
        )


        process_available = torch.stack(
            available_list,
            dim=1
        )


        weights = (
            process_available
            .float()
            .unsqueeze(-1)
        )


        balanced = (

            (
                process_pooled
                * weights
            )
            .sum(
                dim=1
            )

            /

            weights.sum(
                dim=1
            )
            .clamp_min(
                1.0
            )
        )


        balanced = (
            self.process_balance_norm(
                balanced
            )
        )


        return (
            balanced,
            process_pooled,
            process_available
        )


    # Forward

    def forward(
        self,
        batch,
        lambda_booklet_grl=1.0,
        lambda_country_grl=1.0
    ):

        credit = batch[
            "credit_tokens"
        ]

        behaviour = batch[
            "behaviour_tokens"
        ]

        interaction_mask = batch[
            "interaction_mask"
        ]

        process_values = batch[
            "process_values"
        ]

        process_mask = batch[
            "process_mask"
        ]


        batch_size = (
            credit.shape[0]
        )

        device = (
            credit.device
        )


        # Shared encoder metadata

        encoder_meta = (
            self.encoder_metadata(
                batch_size
            )
        )


        # Performance branch

        performance_tokens = (

            encoder_meta

            +

            self.credit_embedding(
                credit
            )

            +

            self.behaviour_embedding(
                behaviour
            )
        )


        performance_encoded = (
            self.performance_encoder(

                performance_tokens,

                src_key_padding_mask=(
                    ~interaction_mask
                )
            )
        )


        # Process branch

        process_input = torch.cat(
            [
                process_values,
                process_mask.float()
            ],
            dim=-1
        )


        process_tokens = (

            encoder_meta

            +

            self.process_projection(
                process_input
            )
        )


        process_item_mask = (
            process_mask.any(
                dim=-1
            )
        )


        safe_process_mask = (
            process_item_mask.clone()
        )


        no_process = (
            ~safe_process_mask.any(
                dim=1
            )
        )


        if no_process.any():

            safe_process_mask[
                no_process,
                0
            ] = True

            process_tokens = (
                process_tokens.clone()
            )

            process_tokens[
                no_process,
                0,
                :
            ] = 0.0


        process_encoded = (
            self.process_encoder(

                process_tokens,

                src_key_padding_mask=(
                    ~safe_process_mask
                )
            )
        )


        # Cross-view fusion

        cross_output, _ = (
            self.cross_attention(

                query=performance_encoded,
                key=process_encoded,
                value=process_encoded,

                key_padding_mask=(
                    ~safe_process_mask
                ),

                need_weights=False
            )
        )


        if no_process.any():

            cross_output = (
                cross_output.clone()
            )

            cross_output[
                no_process
            ] = 0.0


        creative_item_tokens = (
            self.cross_norm(

                performance_encoded

                +

                cross_output
            )
        )


        # Process-balanced z_general

        (
            balanced_pooled,
            ideation_process_pooled,
            ideation_process_available
        ) = self.process_balanced_pool(

            creative_item_tokens,

            interaction_mask
        )


        z_general = (
            self.creative_projection(
                balanced_pooled
            )
        )


        # Context latent

        student_context = (
            self.student_encoder(

                batch[
                    "student_continuous"
                ],

                batch[
                    "student_continuous_mask"
                ],

                batch[
                    "student_categorical"
                ]
            )
        )


        school_context = (
            self.school_encoder(

                batch[
                    "school_continuous"
                ],

                batch[
                    "school_continuous_mask"
                ],

                batch[
                    "school_categorical"
                ]
            )
        )


        context_fused = (
            self.context_fusion(

                torch.cat(
                    [
                        student_context,
                        school_context
                    ],
                    dim=1
                )
            )
        )


        z_context = (
            self.context_projection(
                context_fused
            )
        )


        # Domain-level pooling

        domain_pooled = []
        domain_available = []


        for domain_id in range(
            self.n_domains
        ):

            selector = (

                self.item_domain_ids
                .unsqueeze(0)

                == domain_id
            )


            valid_mask = (

                interaction_mask

                &

                selector
            )


            available = (
                valid_mask.any(
                    dim=1
                )
            )


            pooled = masked_mean(

                creative_item_tokens,

                valid_mask,

                dim=1
            )


            pooled = torch.where(

                available.unsqueeze(-1),

                pooled,

                torch.zeros_like(
                    pooled
                )
            )


            domain_pooled.append(
                pooled
            )

            domain_available.append(
                available
            )


        domain_pooled = torch.stack(
            domain_pooled,
            dim=1
        )


        domain_available = torch.stack(
            domain_available,
            dim=1
        )


        # Shared domain representation

        z_domain_shared = (
            self.domain_shared_projection(
                domain_pooled
            )
        )


        z_domain_shared = torch.where(

            domain_available
            .unsqueeze(-1),

            z_domain_shared,

            torch.zeros_like(
                z_domain_shared
            )
        )


        # Domain-specific representation

        specific_list = []


        for domain_id in range(
            self.n_domains
        ):

            z_specific = (
                self.domain_specific_projections[
                    domain_id
                ](
                    domain_pooled[
                        :,
                        domain_id,
                        :
                    ]
                )
            )


            z_specific = torch.where(

                domain_available[
                    :,
                    domain_id
                ]
                .unsqueeze(-1),

                z_specific,

                torch.zeros_like(
                    z_specific
                )
            )


            specific_list.append(
                z_specific
            )


        z_domain_specific = torch.stack(
            specific_list,
            dim=1
        )


        z_domain_creative = (

            z_general
            .unsqueeze(1)

            +

            z_domain_specific
        )


        z_domain_creative = torch.where(

            domain_available
            .unsqueeze(-1),

            z_domain_creative,

            torch.zeros_like(
                z_domain_creative
            )
        )


        # Decoder-only exact item identity

        decoder_meta = (
            self.decoder_metadata(
                batch_size,
                device
            )
        )


        decoder_item_tokens = (

            creative_item_tokens

            +

            decoder_meta
        )


        credit_logits = (
            self.credit_head(
                decoder_item_tokens
            )
        )


        process_prediction = (
            self.process_head(
                decoder_item_tokens
            )
        )


        # Domain-decomposition reconstruction

        domain_latent_per_item = (
            z_domain_creative
            .index_select(
                dim=1,
                index=self.item_domain_ids
            )
        )


        decomposition_tokens = (

            self.domain_latent_decoder(
                domain_latent_per_item
            )

            +

            decoder_meta
        )


        decomposition_credit_logits = (
            self.decomposition_credit_head(
                decomposition_tokens
            )
        )


        # Context reconstruction

        if (
            self.context_continuous_head
            is not None
        ):

            context_continuous_prediction = (
                self.context_continuous_head(
                    z_context
                )
            )

        else:

            context_continuous_prediction = (
                z_context.new_empty(
                    (
                        batch_size,
                        0
                    )
                )
            )


        context_categorical_logits = [

            head(
                z_context
            )

            for head
            in self.context_categorical_heads
        ]


        # Country / booklet routing

        booklet_logits = (
            self.booklet_classifier(

                grad_reverse(
                    z_general,
                    lambda_booklet_grl
                )
            )
        )


        context_country_logits = (
            self.context_country_classifier(
                z_context
            )
        )


        creative_country_logits = (
            self.creative_country_classifier(

                grad_reverse(
                    z_general,
                    lambda_country_grl
                )
            )
        )


        return {

            "creative_item_tokens":
                creative_item_tokens,

            "z_general":
                z_general,

            "z_creative":
                z_general,

            "z_context":
                z_context,

            "z_domain_shared":
                z_domain_shared,

            "z_domain_specific":
                z_domain_specific,

            "z_domain_creative":
                z_domain_creative,

            "domain_available":
                domain_available,

            "ideation_process_pooled":
                ideation_process_pooled,

            "ideation_process_available":
                ideation_process_available,

            "credit_logits":
                credit_logits,

            "process_prediction":
                process_prediction,

            "decomposition_credit_logits":
                decomposition_credit_logits,

            "context_continuous_prediction":
                context_continuous_prediction,

            "context_categorical_logits":
                context_categorical_logits,

            "booklet_logits":
                booklet_logits,

            "context_country_logits":
                context_country_logits,

            "creative_country_logits":
                creative_country_logits,

            "process_item_mask":
                process_item_mask
        }


print(
    "Standalone CReST V5 architecture ready."
)

Standalone CReST V5 architecture ready.


In [ ]:
# Instantiate and smoke-test standalone CReST V5

student_cardinalities = (
    fold0[
        "student_preprocessor"
    ]
    .categorical_cardinalities()
)


school_cardinalities = (
    fold0[
        "school_preprocessor"
    ]
    .categorical_cardinalities()
)


model = CReSTV5(

    item_domain_ids=(
        item_domain_ids
    ),

    item_ideation_ids=(
        item_ideation_ids
    ),

    student_n_continuous=(
        len(
            fold0[
                "student_preprocessor"
            ]
            .continuous_vars
        )
    ),

    student_cat_cardinalities=(
        student_cardinalities
    ),

    school_n_continuous=(
        len(
            fold0[
                "school_preprocessor"
            ]
            .continuous_vars
        )
    ),

    school_cat_cardinalities=(
        school_cardinalities
    ),

    n_train_countries=(
        len(
            fold0[
                "country_to_train_id"
            ]
        )
    ),

    n_booklets=30,
    n_domains=4,
    n_processes=3,

    d_model=int(
        CONFIG["d_model"]
    ),

    latent_dim=int(
        CONFIG["latent_dim"]
    ),

    n_heads=int(
        CONFIG["n_heads"]
    ),

    n_layers=int(
        CONFIG["n_layers"]
    ),

    dropout=float(
        CONFIG["dropout"]
    )
).to(device)


batch = next(
    iter(
        train_loader
    )
)


batch = {
    key:
        value.to(
            device
        )

    for key, value
    in batch.items()
}


model.eval()


with torch.no_grad():

    output = model(

        batch,

        lambda_booklet_grl=0.1,

        lambda_country_grl=0.1
    )


print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)


print(
    "\nOutput shapes:"
)


for key in [

    "z_general",
    "z_context",
    "z_domain_shared",
    "z_domain_specific",
    "z_domain_creative",
    "ideation_process_pooled",
    "ideation_process_available",
    "credit_logits",
    "process_prediction",
    "decomposition_credit_logits",
    "context_continuous_prediction",
    "booklet_logits",
    "context_country_logits",
    "creative_country_logits"

]:

    value = output[
        key
    ]

    print(
        f"{key:31s}",
        tuple(
            value.shape
        )
    )

Trainable parameters: 1438460

Output shapes:
z_general                       (256, 64)
z_context                       (256, 64)
z_domain_shared                 (256, 4, 64)
z_domain_specific               (256, 4, 64)
z_domain_creative               (256, 4, 64)
ideation_process_pooled         (256, 3, 128)
ideation_process_available      (256, 3)
credit_logits                   (256, 32, 3)
process_prediction              (256, 32, 3)
decomposition_credit_logits     (256, 32, 3)
context_continuous_prediction   (256, 2)
booklet_logits                  (256, 30)
context_country_logits          (256, 45)
creative_country_logits         (256, 45)


In [ ]:
# Rebuild CReST V5 with explicit categorical MASK tokens

student_base_cardinalities = (
    fold0["student_preprocessor"]
    .categorical_cardinalities()
)

school_base_cardinalities = (
    fold0["school_preprocessor"]
    .categorical_cardinalities()
)


# Existing valid IDs are 0 ... cardinality-1.
# New SSL MASK token = cardinality.

student_context_mask_ids = (
    student_base_cardinalities.copy()
)

school_context_mask_ids = (
    school_base_cardinalities.copy()
)


# Encoder embedding tables need one additional token.
student_model_cardinalities = [
    c + 1
    for c in student_base_cardinalities
]

school_model_cardinalities = [
    c + 1
    for c in school_base_cardinalities
]


set_global_seed()


model = CReSTV5(

    item_domain_ids=item_domain_ids,
    item_ideation_ids=item_ideation_ids,

    student_n_continuous=len(
        fold0[
            "student_preprocessor"
        ].continuous_vars
    ),

    student_cat_cardinalities=(
        student_model_cardinalities
    ),

    school_n_continuous=len(
        fold0[
            "school_preprocessor"
        ].continuous_vars
    ),

    school_cat_cardinalities=(
        school_model_cardinalities
    ),

    n_train_countries=len(
        fold0[
            "country_to_train_id"
        ]
    ),

    n_booklets=30,
    n_domains=4,
    n_processes=3,

    d_model=int(
        CONFIG["d_model"]
    ),

    latent_dim=int(
        CONFIG["latent_dim"]
    ),

    n_heads=int(
        CONFIG["n_heads"]
    ),

    n_layers=int(
        CONFIG["n_layers"]
    ),

    dropout=float(
        CONFIG["dropout"]
    )
).to(device)


print(
    "Student base cardinalities:",
    student_base_cardinalities
)

print(
    "Student SSL mask IDs:",
    student_context_mask_ids
)

print(
    "School base cardinalities:",
    school_base_cardinalities
)

print(
    "School SSL mask IDs:",
    school_context_mask_ids
)

print(
    "\nTrainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)

Student base cardinalities: [4, 4, 5, 4]
Student SSL mask IDs: [4, 4, 5, 4]
School base cardinalities: [8, 7, 9]
School SSL mask IDs: [8, 7, 9]

Trainable parameters: 1439027


In [ ]:
# Self-supervised corruption functions

def ensure_one_mask_per_eligible_row(
    selected_mask,
    eligible_mask
):

    needs_one = (
        eligible_mask.any(dim=1)
        &
        ~selected_mask.any(dim=1)
    )

    for row in torch.where(
        needs_one
    )[0]:

        positions = torch.where(
            eligible_mask[row]
        )[0]

        choice = positions[
            torch.randint(
                len(positions),
                (1,),
                device=positions.device
            )
        ]

        selected_mask[
            row,
            choice
        ] = True

    return selected_mask


def make_main_ssl_corruption(
    batch,
    credit_mask_prob=0.15,
    process_mask_prob=0.15
):

    corrupted = {
        k: v.clone()
        for k, v in batch.items()
    }


    # Credit masking

    credit_eligible = (
        batch["performance_mask"]
    )

    credit_ssl_mask = (
        credit_eligible
        &
        (
            torch.rand(
                credit_eligible.shape,
                device=credit_eligible.device
            )
            < credit_mask_prob
        )
    )

    credit_ssl_mask = (
        ensure_one_mask_per_eligible_row(
            credit_ssl_mask,
            credit_eligible
        )
    )


    # Original tokens 1/2/3 -> classes 0/1/2
    credit_targets = (
        batch["credit_tokens"]
        - 1
    )


    # Dedicated credit SSL MASK token
    corrupted[
        "credit_tokens"
    ][
        credit_ssl_mask
    ] = 4


    # Process masking

    process_item_eligible = (
        batch[
            "process_mask"
        ].any(
            dim=-1
        )
    )


    process_ssl_item_mask = (
        process_item_eligible
        &
        (
            torch.rand(
                process_item_eligible.shape,
                device=process_item_eligible.device
            )
            < process_mask_prob
        )
    )


    process_ssl_item_mask = (
        ensure_one_mask_per_eligible_row(
            process_ssl_item_mask,
            process_item_eligible
        )
    )


    process_targets = (
        batch[
            "process_values"
        ].clone()
    )


    expanded = (
        process_ssl_item_mask
        .unsqueeze(-1)
        .expand_as(
            batch[
                "process_values"
            ]
        )
    )


    process_reconstruction_mask = (
        expanded
        &
        batch[
            "process_mask"
        ]
    )


    corrupted[
        "process_values"
    ][
        expanded
    ] = 0.0


    corrupted[
        "process_mask"
    ][
        expanded
    ] = False


    return {
        "corrupted_batch":
            corrupted,

        "credit_targets":
            credit_targets,

        "credit_ssl_mask":
            credit_ssl_mask,

        "process_targets":
            process_targets,

        "process_reconstruction_mask":
            process_reconstruction_mask
    }


def make_context_ssl_corruption(
    batch,
    mask_prob=0.20
):

    corrupted = {
        k: v.clone()
        for k, v in batch.items()
    }


    continuous_targets = []
    continuous_masks = []

    categorical_targets = []
    categorical_masks = []


    # Continuous context

    for prefix in [
        "student",
        "school"
    ]:

        values = batch[
            f"{prefix}_continuous"
        ]

        observed = batch[
            f"{prefix}_continuous_mask"
        ]


        selected = (
            observed
            &
            (
                torch.rand(
                    observed.shape,
                    device=observed.device
                )
                < mask_prob
            )
        )


        continuous_targets.append(
            values.clone()
        )

        continuous_masks.append(
            selected
        )


        corrupted[
            f"{prefix}_continuous"
        ][
            selected
        ] = 0.0


        corrupted[
            f"{prefix}_continuous_mask"
        ][
            selected
        ] = False


    # Categorical context

    categorical_spec = [
        (
            "student",
            student_context_mask_ids
        ),
        (
            "school",
            school_context_mask_ids
        )
    ]


    for prefix, mask_ids in categorical_spec:

        values = batch[
            f"{prefix}_categorical"
        ]

        # 0 = missing, 1 = unseen.
        # Only actual observed categories are masked.
        observed = (
            values >= 2
        )


        selected = (
            observed
            &
            (
                torch.rand(
                    observed.shape,
                    device=observed.device
                )
                < mask_prob
            )
        )


        categorical_targets.append(
            values.clone()
        )

        categorical_masks.append(
            selected
        )


        for j, mask_id in enumerate(
            mask_ids
        ):

            feature_selected = (
                selected[:, j]
            )

            corrupted[
                f"{prefix}_categorical"
            ][
                feature_selected,
                j
            ] = int(
                mask_id
            )


    return {
        "corrupted_batch":
            corrupted,

        "continuous_targets":
            torch.cat(
                continuous_targets,
                dim=1
            ),

        "continuous_ssl_mask":
            torch.cat(
                continuous_masks,
                dim=1
            ),

        "categorical_targets":
            torch.cat(
                categorical_targets,
                dim=1
            ),

        "categorical_ssl_mask":
            torch.cat(
                categorical_masks,
                dim=1
            )
    }


print(
    "SSL corruption functions ready."
)

SSL corruption functions ready.


In [ ]:
# CReST V5 objective functions

def masked_credit_loss(
    logits,
    targets,
    mask
):

    if not mask.any():
        return logits.sum() * 0.0

    return F.cross_entropy(
        logits[mask],
        targets[mask]
    )


def masked_process_loss(
    prediction,
    targets,
    mask
):

    if not mask.any():
        return prediction.sum() * 0.0

    return F.smooth_l1_loss(
        prediction[mask],
        targets[mask],
        beta=1.0
    )


def context_continuous_loss(
    prediction,
    targets,
    mask
):

    if not mask.any():
        return prediction.sum() * 0.0

    return F.smooth_l1_loss(
        prediction[mask],
        targets[mask],
        beta=1.0
    )


def context_categorical_loss(
    logits_list,
    targets,
    mask
):

    losses = []

    for j, logits in enumerate(
        logits_list
    ):

        selected = mask[:, j]

        if selected.any():

            losses.append(
                F.cross_entropy(
                    logits[selected],
                    targets[
                        selected,
                        j
                    ]
                )
            )


    if not losses:
        return logits_list[0].sum() * 0.0

    return torch.stack(
        losses
    ).mean()


def cross_latent_decorrelation(
    z_general,
    z_context,
    eps=1e-6
):

    zg = (
        z_general
        -
        z_general.mean(
            dim=0,
            keepdim=True
        )
    )

    zc = (
        z_context
        -
        z_context.mean(
            dim=0,
            keepdim=True
        )
    )


    zg = zg / (
        zg.std(
            dim=0,
            unbiased=False,
            keepdim=True
        )
        + eps
    )

    zc = zc / (
        zc.std(
            dim=0,
            unbiased=False,
            keepdim=True
        )
        + eps
    )


    corr = (
        zg.T @ zc
    ) / zg.shape[0]

    return corr.pow(2).mean()


def variance_loss(
    z,
    target_std=1.0
):

    std = torch.sqrt(
        z.var(
            dim=0,
            unbiased=False
        )
        + 1e-4
    )

    return F.relu(
        target_std - std
    ).mean()


def domain_alignment_loss(
    z_general,
    z_shared,
    available
):

    general = (
        z_general
        .unsqueeze(1)
        .expand_as(
            z_shared
        )
    )

    cosine = F.cosine_similarity(
        general[available],
        z_shared[available],
        dim=-1
    )

    return (
        1.0 - cosine
    ).mean()


def domain_orthogonality_loss(
    z_general,
    z_specific,
    available
):

    general = (
        z_general
        .unsqueeze(1)
        .expand_as(
            z_specific
        )
    )

    cosine = F.cosine_similarity(
        general[available],
        z_specific[available],
        dim=-1
    )

    return cosine.pow(2).mean()


def domain_variance_loss(
    z_specific,
    available,
    target_std=0.5
):

    losses = []

    for d in range(
        z_specific.shape[1]
    ):

        valid = available[:, d]

        if valid.sum() < 2:
            continue

        std = torch.sqrt(
            z_specific[
                valid,
                d
            ].var(
                dim=0,
                unbiased=False
            )
            + 1e-4
        )

        losses.append(
            F.relu(
                target_std - std
            ).mean()
        )


    if not losses:
        return z_specific.sum() * 0.0

    return torch.stack(
        losses
    ).mean()


print(
    "CReST V5 loss functions ready."
)

CReST V5 loss functions ready.


In [ ]:
# Composite objective and backward smoke test

LOSS_WEIGHTS = {
    "credit": 1.00,
    "process": 0.50,
    "decomposition_credit": 0.25,
    "context": 0.50,
    "decorrelation": 0.10,
    "booklet": 0.10,
    "country_context": 0.10,
    "country_creative": 0.10,
    "global_variance": 0.05,
    "domain_alignment": 0.05,
    "domain_orthogonality": 0.05,
    "domain_variance": 0.02
}


def compute_crest_v5_objective(
    model,
    batch,
    lambda_booklet_grl=1.0,
    lambda_country_grl=1.0,
    credit_mask_prob=0.15,
    process_mask_prob=0.15,
    context_mask_prob=0.20
):

    # Main-view SSL corruption

    main_ssl = make_main_ssl_corruption(
        batch,
        credit_mask_prob=credit_mask_prob,
        process_mask_prob=process_mask_prob
    )


    # Context SSL corruption
    #
    # Applied on top of main-view corruption so model sees
    # all masked views simultaneously.

    context_ssl = make_context_ssl_corruption(
        main_ssl["corrupted_batch"],
        mask_prob=context_mask_prob
    )


    corrupted_batch = (
        context_ssl["corrupted_batch"]
    )


    # Forward

    output = model(

        corrupted_batch,

        lambda_booklet_grl=(
            lambda_booklet_grl
        ),

        lambda_country_grl=(
            lambda_country_grl
        )
    )


    # 1. Masked credit reconstruction

    loss_credit = masked_credit_loss(

        output["credit_logits"],

        main_ssl[
            "credit_targets"
        ],

        main_ssl[
            "credit_ssl_mask"
        ]
    )


    # 2. Masked process reconstruction

    loss_process = masked_process_loss(

        output[
            "process_prediction"
        ],

        main_ssl[
            "process_targets"
        ],

        main_ssl[
            "process_reconstruction_mask"
        ]
    )


    # 3. Domain-decomposition credit reconstruction

    loss_decomposition_credit = (
        masked_credit_loss(

            output[
                "decomposition_credit_logits"
            ],

            main_ssl[
                "credit_targets"
            ],

            main_ssl[
                "credit_ssl_mask"
            ]
        )
    )


    # 4. Context reconstruction

    loss_context_continuous = (
        context_continuous_loss(

            output[
                "context_continuous_prediction"
            ],

            context_ssl[
                "continuous_targets"
            ],

            context_ssl[
                "continuous_ssl_mask"
            ]
        )
    )


    loss_context_categorical = (
        context_categorical_loss(

            output[
                "context_categorical_logits"
            ],

            context_ssl[
                "categorical_targets"
            ],

            context_ssl[
                "categorical_ssl_mask"
            ]
        )
    )


    # Equal contribution from continuous and categorical
    # context reconstruction.
    loss_context = 0.5 * (
        loss_context_continuous
        +
        loss_context_categorical
    )


    # 5. Creative-context decorrelation

    loss_decorrelation = (
        cross_latent_decorrelation(

            output[
                "z_general"
            ],

            output[
                "z_context"
            ]
        )
    )


    # 6. Booklet adversarial loss

    loss_booklet = F.cross_entropy(

        output[
            "booklet_logits"
        ],

        batch[
            "booklet_nuisance_id"
        ]
    )


    # 7. Country routing

    country_targets = (
        batch[
            "country_train_id"
        ]
    )


    loss_country_context = (
        F.cross_entropy(

            output[
                "context_country_logits"
            ],

            country_targets
        )
    )


    loss_country_creative = (
        F.cross_entropy(

            output[
                "creative_country_logits"
            ],

            country_targets
        )
    )


    # 8. Global latent variance

    loss_global_variance = (
        variance_loss(
            output[
                "z_general"
            ]
        )
    )


    # 9. Domain alignment

    loss_domain_alignment = (
        domain_alignment_loss(

            output[
                "z_general"
            ],

            output[
                "z_domain_shared"
            ],

            output[
                "domain_available"
            ]
        )
    )


    # 10. General-specific orthogonality

    loss_domain_orthogonality = (
        domain_orthogonality_loss(

            output[
                "z_general"
            ],

            output[
                "z_domain_specific"
            ],

            output[
                "domain_available"
            ]
        )
    )


    # 11. Domain-specific variance

    loss_domain_variance = (
        domain_variance_loss(

            output[
                "z_domain_specific"
            ],

            output[
                "domain_available"
            ]
        )
    )


    # Total objective

    losses = {

        "credit":
            loss_credit,

        "process":
            loss_process,

        "decomposition_credit":
            loss_decomposition_credit,

        "context":
            loss_context,

        "context_continuous":
            loss_context_continuous,

        "context_categorical":
            loss_context_categorical,

        "decorrelation":
            loss_decorrelation,

        "booklet":
            loss_booklet,

        "country_context":
            loss_country_context,

        "country_creative":
            loss_country_creative,

        "global_variance":
            loss_global_variance,

        "domain_alignment":
            loss_domain_alignment,

        "domain_orthogonality":
            loss_domain_orthogonality,

        "domain_variance":
            loss_domain_variance
    }


    total_loss = sum(

        LOSS_WEIGHTS[name]
        *
        losses[name]

        for name
        in LOSS_WEIGHTS
    )


    return (
        total_loss,
        losses,
        output
    )


# Small smoke-test batch

smoke_batch = next(
    iter(
        train_loader
    )
)


SMOKE_N = min(
    64,
    smoke_batch[
        "credit_tokens"
    ].shape[0]
)


smoke_batch = {

    key:
        value[:SMOKE_N]
        .to(device)

    for key, value
    in smoke_batch.items()
}


# Forward + backward

set_global_seed(
    GLOBAL_SEED + 10
)

model.train()
model.zero_grad(
    set_to_none=True
)


total_loss, smoke_losses, smoke_output = (
    compute_crest_v5_objective(

        model,
        smoke_batch,

        lambda_booklet_grl=0.5,
        lambda_country_grl=0.5
    )
)


assert torch.isfinite(
    total_loss
), "Total loss is not finite."


total_loss.backward()


# Gradient audit

def grad_norm(
    parameter
):

    if parameter.grad is None:
        return float("nan")

    return float(
        parameter.grad
        .detach()
        .norm()
        .cpu()
    )


critical_gradients = {

    "item_embedding_decoder_only":
        grad_norm(
            model.item_embedding.weight
        ),

    "process_balance_norm":
        grad_norm(
            model.process_balance_norm.weight
        ),

    "creative_projection":
        grad_norm(
            model.creative_projection[0].weight
        ),

    "context_projection":
        grad_norm(
            model.context_projection[0].weight
        ),

    "booklet_classifier":
        grad_norm(
            model.booklet_classifier[0].weight
        ),

    "creative_country_classifier":
        grad_norm(
            model.creative_country_classifier[0].weight
        )
}


# Report

print(
    "Total loss:",
    round(
        float(
            total_loss.detach().cpu()
        ),
        6
    )
)


print(
    "\nLoss components:"
)

for name, value in smoke_losses.items():

    print(
        f"{name:26s}",
        f"{float(value.detach().cpu()):.6f}"
    )


print(
    "\nCritical gradient norms:"
)

for name, value in critical_gradients.items():

    print(
        f"{name:31s}",
        f"{value:.6f}"
    )


assert all(
    np.isfinite(v)
    and v > 0

    for v
    in critical_gradients.values()
)


print(
    "\nCReST V5 backward smoke test: PASS"
)

Total loss: 4.063176

Loss components:
credit                     1.450437
process                    0.849637
decomposition_credit       1.489727
context                    1.245565
context_continuous         0.361279
context_categorical        2.129850
decorrelation              0.015355
booklet                    3.384815
country_context            3.820527
country_creative           3.857204
global_variance            0.679065
domain_alignment           0.933352
domain_orthogonality       0.009043
domain_variance            0.192148

Critical gradient norms:
item_embedding_decoder_only     0.078208
process_balance_norm            0.017962
creative_projection             0.391641
context_projection              0.583135
booklet_classifier              0.055824
creative_country_classifier     0.037664

CReST V5 backward smoke test: PASS


In [ ]:
# Final CReST V5 model factory
# Encoder has explicit MASK tokens; decoder predicts only real/base classes.

def build_crest_v5_model(
    seed=GLOBAL_SEED
):

    set_global_seed(
        seed
    )

    m = CReSTV5(

        item_domain_ids=item_domain_ids,
        item_ideation_ids=item_ideation_ids,

        student_n_continuous=len(
            fold0[
                "student_preprocessor"
            ].continuous_vars
        ),

        # +1 only for encoder MASK token
        student_cat_cardinalities=(
            student_model_cardinalities
        ),

        school_n_continuous=len(
            fold0[
                "school_preprocessor"
            ].continuous_vars
        ),

        school_cat_cardinalities=(
            school_model_cardinalities
        ),

        n_train_countries=len(
            fold0[
                "country_to_train_id"
            ]
        ),

        n_booklets=30,
        n_domains=4,
        n_processes=3,

        d_model=int(
            CONFIG["d_model"]
        ),

        latent_dim=int(
            CONFIG["latent_dim"]
        ),

        n_heads=int(
            CONFIG["n_heads"]
        ),

        n_layers=int(
            CONFIG["n_layers"]
        ),

        dropout=float(
            CONFIG["dropout"]
        )
    )


    # Reconstruction heads use BASE cardinalities only.
    # The SSL MASK token is therefore never a target class.

    target_cardinalities = (
        student_base_cardinalities
        +
        school_base_cardinalities
    )


    m.context_categorical_heads = nn.ModuleList(
        [
            nn.Linear(
                int(CONFIG["latent_dim"]),
                cardinality
            )

            for cardinality
            in target_cardinalities
        ]
    )


    return m.to(
        device
    )


model = build_crest_v5_model()


# Quick shape verification

check_batch = next(
    iter(
        train_loader
    )
)

check_batch = {
    k: v[:32].to(device)
    for k, v
    in check_batch.items()
}


model.eval()

with torch.no_grad():

    check_out = model(
        check_batch,
        lambda_booklet_grl=0.0,
        lambda_country_grl=0.0
    )


print(
    "Context reconstruction output sizes:"
)

print(
    [
        x.shape[1]
        for x
        in check_out[
            "context_categorical_logits"
        ]
    ]
)

print(
    "Expected:",
    student_base_cardinalities
    +
    school_base_cardinalities
)

print(
    "\nFinal trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)

Context reconstruction output sizes:
[4, 4, 5, 4, 8, 7, 9]
Expected: [4, 4, 5, 4, 8, 7, 9]

Final trainable parameters: 1438572


In [ ]:
# Pilot training and country-equal validation utilities

PILOT_EPOCHS = 5

CHECKPOINT_FOLD0_DIR = (
    CHECKPOINT_DIR
    / "fold_0"
)

CHECKPOINT_FOLD0_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def grl_schedule(
    progress
):

    return (
        2.0
        /
        (
            1.0
            +
            np.exp(
                -10.0 * progress
            )
        )
        -
        1.0
    )


def reconstruction_score_from_losses(
    losses
):

    return (

        losses["credit"]

        +

        LOSS_WEIGHTS[
            "process"
        ]
        *
        losses["process"]

        +

        LOSS_WEIGHTS[
            "decomposition_credit"
        ]
        *
        losses[
            "decomposition_credit"
        ]

        +

        LOSS_WEIGHTS[
            "context"
        ]
        *
        losses["context"]
    )


@torch.no_grad()
def evaluate_country_equal_reconstruction(
    model,
    seed=GLOBAL_SEED + 500
):

    model.eval()

    val_countries = (
        fold0[
            "validation"
        ]["CNT"]
    )

    country_scores = {}


    # Fixed corruption pattern across epochs
    with torch.random.fork_rng():

        torch.manual_seed(
            seed
        )


        for cnt in sorted(
            np.unique(
                val_countries
            )
        ):

            local_idx = np.where(
                val_countries
                == cnt
            )[0]


            loader = DataLoader(

                Subset(
                    validation_dataset,
                    local_idx.tolist()
                ),

                batch_size=BATCH_SIZE,

                shuffle=False,

                num_workers=0
            )


            weighted_sum = 0.0
            n_students = 0


            for batch in loader:

                batch = {
                    k: v.to(device)
                    for k, v
                    in batch.items()
                }


                main_ssl = (
                    make_main_ssl_corruption(
                        batch,
                        credit_mask_prob=0.15,
                        process_mask_prob=0.15
                    )
                )


                context_ssl = (
                    make_context_ssl_corruption(
                        main_ssl[
                            "corrupted_batch"
                        ],
                        mask_prob=0.20
                    )
                )


                out = model(

                    context_ssl[
                        "corrupted_batch"
                    ],

                    lambda_booklet_grl=0.0,
                    lambda_country_grl=0.0
                )


                val_losses = {}


                val_losses["credit"] = (
                    masked_credit_loss(

                        out[
                            "credit_logits"
                        ],

                        main_ssl[
                            "credit_targets"
                        ],

                        main_ssl[
                            "credit_ssl_mask"
                        ]
                    )
                )


                val_losses["process"] = (
                    masked_process_loss(

                        out[
                            "process_prediction"
                        ],

                        main_ssl[
                            "process_targets"
                        ],

                        main_ssl[
                            "process_reconstruction_mask"
                        ]
                    )
                )


                val_losses[
                    "decomposition_credit"
                ] = masked_credit_loss(

                    out[
                        "decomposition_credit_logits"
                    ],

                    main_ssl[
                        "credit_targets"
                    ],

                    main_ssl[
                        "credit_ssl_mask"
                    ]
                )


                c_cont = (
                    context_continuous_loss(

                        out[
                            "context_continuous_prediction"
                        ],

                        context_ssl[
                            "continuous_targets"
                        ],

                        context_ssl[
                            "continuous_ssl_mask"
                        ]
                    )
                )


                c_cat = (
                    context_categorical_loss(

                        out[
                            "context_categorical_logits"
                        ],

                        context_ssl[
                            "categorical_targets"
                        ],

                        context_ssl[
                            "categorical_ssl_mask"
                        ]
                    )
                )


                val_losses["context"] = (
                    0.5
                    *
                    (
                        c_cont
                        +
                        c_cat
                    )
                )


                score = (
                    reconstruction_score_from_losses(
                        val_losses
                    )
                )


                n_batch = (
                    batch[
                        "credit_tokens"
                    ].shape[0]
                )


                weighted_sum += (
                    float(
                        score.cpu()
                    )
                    *
                    n_batch
                )

                n_students += (
                    n_batch
                )


            country_scores[cnt] = (
                weighted_sum
                /
                n_students
            )


    country_equal = float(
        np.mean(
            list(
                country_scores.values()
            )
        )
    )


    return (
        country_equal,
        country_scores
    )


print(
    "Pilot training utilities ready."
)

Pilot training utilities ready.


In [ ]:
# Train CReST V5 Fold-0 pilot

model = build_crest_v5_model(
    seed=GLOBAL_SEED
)


optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=1e-4,

    weight_decay=1e-4
)


scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(

        optimizer,

        mode="min",

        factor=0.5,

        patience=2,

        min_lr=1e-6
    )
)


pilot_history = []

best_val_score = np.inf
best_epoch = None

pilot_checkpoint = (
    CHECKPOINT_FOLD0_DIR
    / "crest_v5_pilot_best.pt"
)


for epoch in range(
    1,
    PILOT_EPOCHS + 1
):

    model.train()

    running_total = 0.0
    running_recon = 0.0

    running_booklet = 0.0
    running_country_context = 0.0
    running_country_creative = 0.0

    n_seen = 0


    for batch_idx, batch in enumerate(
        train_loader
    ):

        batch = {
            k: v.to(device)
            for k, v
            in batch.items()
        }


        progress = (

            (
                epoch - 1
            )

            +

            batch_idx
            /
            max(
                len(
                    train_loader
                ),
                1
            )
        ) / PILOT_EPOCHS


        lambda_grl = float(
            grl_schedule(
                progress
            )
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        total_loss, losses, _ = (
            compute_crest_v5_objective(

                model,
                batch,

                lambda_booklet_grl=(
                    lambda_grl
                ),

                lambda_country_grl=(
                    lambda_grl
                )
            )
        )


        total_loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )


        optimizer.step()


        n_batch = (
            batch[
                "credit_tokens"
            ].shape[0]
        )


        recon = (
            reconstruction_score_from_losses(
                losses
            )
        )


        running_total += (
            float(
                total_loss.detach().cpu()
            )
            *
            n_batch
        )

        running_recon += (
            float(
                recon.detach().cpu()
            )
            *
            n_batch
        )

        running_booklet += (
            float(
                losses[
                    "booklet"
                ].detach().cpu()
            )
            *
            n_batch
        )

        running_country_context += (
            float(
                losses[
                    "country_context"
                ].detach().cpu()
            )
            *
            n_batch
        )

        running_country_creative += (
            float(
                losses[
                    "country_creative"
                ].detach().cpu()
            )
            *
            n_batch
        )

        n_seen += (
            n_batch
        )


    # Country-equal unseen-system validation

    val_score, val_country_scores = (
        evaluate_country_equal_reconstruction(
            model
        )
    )


    scheduler.step(
        val_score
    )


    lr_now = (
        optimizer.param_groups[
            0
        ]["lr"]
    )


    epoch_row = {

        "epoch":
            epoch,

        "train_total":
            running_total
            / n_seen,

        "train_reconstruction":
            running_recon
            / n_seen,

        "train_booklet_ce":
            running_booklet
            / n_seen,

        "train_country_context_ce":
            running_country_context
            / n_seen,

        "train_country_creative_ce":
            running_country_creative
            / n_seen,

        "val_country_equal_reconstruction":
            val_score,

        "learning_rate":
            lr_now
    }


    pilot_history.append(
        epoch_row
    )


    if val_score < best_val_score:

        best_val_score = (
            val_score
        )

        best_epoch = (
            epoch
        )


        torch.save(
            {
                "epoch":
                    epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "val_country_equal_reconstruction":
                    val_score,

                "val_country_scores":
                    val_country_scores
            },
            pilot_checkpoint
        )


    print(

        f"Epoch {epoch:02d} | "

        f"train={epoch_row['train_total']:.4f} | "

        f"recon={epoch_row['train_reconstruction']:.4f} | "

        f"val={val_score:.4f} | "

        f"bookletCE={epoch_row['train_booklet_ce']:.4f} | "

        f"ctxCountryCE={epoch_row['train_country_context_ce']:.4f} | "

        f"creativeCountryCE={epoch_row['train_country_creative_ce']:.4f} | "

        f"GRL={lambda_grl:.3f} | "

        f"lr={lr_now:.1e}"
    )


pilot_history_df = pd.DataFrame(
    pilot_history
)


pilot_history_df.to_csv(

    RESULTS_DIR
    / "fold0_v5_pilot_history.csv",

    index=False
)


print(
    "\nBest pilot epoch:",
    best_epoch
)

print(
    "Best country-equal validation reconstruction:",
    round(
        best_val_score,
        6
    )
)

print(
    "Checkpoint:",
    pilot_checkpoint
)

Epoch 01 | train=2.7785 | recon=1.6677 | val=1.4951 | bookletCE=3.3066 | ctxCountryCE=3.7215 | creativeCountryCE=3.7596 | GRL=0.761 | lr=1.0e-04
Epoch 02 | train=2.5614 | recon=1.5030 | val=1.4536 | bookletCE=3.2955 | ctxCountryCE=3.2811 | creativeCountryCE=3.7979 | GRL=0.964 | lr=1.0e-04
Epoch 03 | train=2.4852 | recon=1.4754 | val=1.4397 | bookletCE=3.1160 | ctxCountryCE=2.9865 | creativeCountryCE=3.8242 | GRL=0.995 | lr=1.0e-04
Epoch 04 | train=2.4385 | recon=1.4613 | val=1.4343 | bookletCE=2.9878 | ctxCountryCE=2.8446 | creativeCountryCE=3.7944 | GRL=0.999 | lr=1.0e-04
Epoch 05 | train=2.4204 | recon=1.4553 | val=1.4271 | bookletCE=2.9287 | ctxCountryCE=2.7809 | creativeCountryCE=3.8227 | GRL=1.000 | lr=1.0e-04

Best pilot epoch: 5
Best country-equal validation reconstruction: 1.427088
Checkpoint: PROJECT_ROOT\checkpoints\fold_0\crest_v5_pilot_best.pt


In [ ]:
# Extract frozen V5 latents from Train and Validation only

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)


# Restore best pilot checkpoint

model = build_crest_v5_model(
    seed=GLOBAL_SEED
)

checkpoint = torch.load(
    pilot_checkpoint,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()


print(
    "Restored pilot epoch:",
    checkpoint["epoch"]
)

print(
    "Validation reconstruction:",
    round(
        checkpoint[
            "val_country_equal_reconstruction"
        ],
        6
    )
)


# Deterministic loaders

PROBE_BATCH_SIZE = 512


probe_train_loader = DataLoader(
    train_dataset,
    batch_size=PROBE_BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


probe_validation_loader = DataLoader(
    validation_dataset,
    batch_size=PROBE_BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


# Frozen representation extraction

@torch.no_grad()
def extract_frozen_latents(
    model,
    loader
):

    general_list = []
    context_list = []
    booklet_list = []

    for batch in loader:

        batch = {
            k: v.to(device)
            for k, v
            in batch.items()
        }

        out = model(
            batch,
            lambda_booklet_grl=0.0,
            lambda_country_grl=0.0
        )

        general_list.append(
            out["z_general"]
            .cpu()
            .numpy()
        )

        context_list.append(
            out["z_context"]
            .cpu()
            .numpy()
        )

        booklet_list.append(
            batch[
                "booklet_nuisance_id"
            ]
            .cpu()
            .numpy()
        )


    return {
        "z_general":
            np.concatenate(
                general_list,
                axis=0
            ),

        "z_context":
            np.concatenate(
                context_list,
                axis=0
            ),

        "booklet":
            np.concatenate(
                booklet_list,
                axis=0
            )
    }


train_latents = extract_frozen_latents(
    model,
    probe_train_loader
)

validation_latents = extract_frozen_latents(
    model,
    probe_validation_loader
)


print(
    "\nTrain z_general:",
    train_latents[
        "z_general"
    ].shape
)

print(
    "Validation z_general:",
    validation_latents[
        "z_general"
    ].shape
)

print(
    "Train booklet classes:",
    len(
        np.unique(
            train_latents[
                "booklet"
            ]
        )
    )
)

print(
    "Validation booklet classes:",
    len(
        np.unique(
            validation_latents[
                "booklet"
            ]
        )
    )
)

Restored pilot epoch: 5
Validation reconstruction: 1.427088

Train z_general: (99966, 64)
Validation z_general: (20949, 64)
Train booklet classes: 30
Validation booklet classes: 30


In [ ]:
# Frozen cross-system booklet probe

def run_booklet_probe(
    X_train,
    y_train,
    X_validation,
    y_validation,
    latent_name
):

    probe = make_pipeline(

        StandardScaler(),

        LogisticRegression(
            max_iter=500,
            solver="lbfgs",
            class_weight="balanced",
            random_state=GLOBAL_SEED
        )
    )


    probe.fit(
        X_train,
        y_train
    )


    pred = probe.predict(
        X_validation
    )


    result = {
        "latent":
            latent_name,

        "accuracy":
            accuracy_score(
                y_validation,
                pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_validation,
                pred
            ),

        "macro_f1":
            f1_score(
                y_validation,
                pred,
                average="macro"
            )
    }


    return result


probe_results = []


probe_results.append(
    run_booklet_probe(

        train_latents[
            "z_general"
        ],

        train_latents[
            "booklet"
        ],

        validation_latents[
            "z_general"
        ],

        validation_latents[
            "booklet"
        ],

        "z_general"
    )
)


probe_results.append(
    run_booklet_probe(

        train_latents[
            "z_context"
        ],

        train_latents[
            "booklet"
        ],

        validation_latents[
            "z_context"
        ],

        validation_latents[
            "booklet"
        ],

        "z_context"
    )
)


booklet_probe_df = pd.DataFrame(
    probe_results
)


booklet_probe_df[
    "random_balanced_accuracy"
] = (
    1.0 / 30.0
)


booklet_probe_df.to_csv(
    RESULTS_DIR
    / "fold0_v5_pilot_booklet_probe.csv",
    index=False
)


print(
    booklet_probe_df.to_string(
        index=False
    )
)


print(
    "\nV3 z_general reference BA: 0.3389"
)

print(
    "Random 30-class reference BA:",
    round(
        1 / 30,
        4
    )
)

   latent  accuracy  balanced_accuracy  macro_f1  random_balanced_accuracy
z_general  0.330660           0.330203  0.233317                  0.033333
z_context  0.006874           0.032836  0.007844                  0.033333

V3 z_general reference BA: 0.3389
Random 30-class reference BA: 0.0333


In [ ]:
# Diagnose structural booklet leakage
# No neural-network training is performed here.

domain_ids_np = np.asarray(
    item_domain_ids,
    dtype=int
)

process_ids_np = np.asarray(
    item_ideation_ids,
    dtype=int
)


# Build count-based design signatures

def build_exposure_features(
    mask
):

    mask = np.asarray(
        mask,
        dtype=np.float32
    )


    # Ideation-process counts: 3 features

    process_counts = np.column_stack(
        [
            mask[
                :,
                process_ids_np == p
            ].sum(
                axis=1
            )

            for p in range(3)
        ]
    )


    # Domain counts: 4 features

    domain_counts = np.column_stack(
        [
            mask[
                :,
                domain_ids_np == d
            ].sum(
                axis=1
            )

            for d in range(4)
        ]
    )


    # Domain x process counts: 12 features

    domain_process_counts = np.column_stack(
        [
            mask[
                :,
                (
                    (domain_ids_np == d)
                    &
                    (process_ids_np == p)
                )
            ].sum(
                axis=1
            )

            for d in range(4)
            for p in range(3)
        ]
    )


    return {
        "process_counts":
            process_counts,

        "domain_counts":
            domain_counts,

        "domain_process_counts":
            domain_process_counts
    }


train_design_features = (
    build_exposure_features(
        fold0[
            "train"
        ][
            "design_mask"
        ]
    )
)

val_design_features = (
    build_exposure_features(
        fold0[
            "validation"
        ][
            "design_mask"
        ]
    )
)


train_interaction_features = (
    build_exposure_features(
        fold0[
            "train"
        ][
            "interaction_mask"
        ]
    )
)

val_interaction_features = (
    build_exposure_features(
        fold0[
            "validation"
        ][
            "interaction_mask"
        ]
    )
)


y_train = (
    fold0[
        "train"
    ][
        "booklet_nuisance_id"
    ]
)

y_val = (
    fold0[
        "validation"
    ][
        "booklet_nuisance_id"
    ]
)


# Cross-system linear probes

def quick_probe(
    X_train,
    X_val,
    name
):

    clf = make_pipeline(

        StandardScaler(),

        LogisticRegression(
            max_iter=500,
            class_weight="balanced",
            random_state=GLOBAL_SEED
        )
    )


    clf.fit(
        X_train,
        y_train
    )


    pred = clf.predict(
        X_val
    )


    return {
        "feature_set":
            name,

        "n_features":
            X_train.shape[1],

        "accuracy":
            accuracy_score(
                y_val,
                pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_val,
                pred
            ),

        "macro_f1":
            f1_score(
                y_val,
                pred,
                average="macro"
            )
    }


diagnostic_results = []


for feature_name in [
    "process_counts",
    "domain_counts",
    "domain_process_counts"
]:

    diagnostic_results.append(

        quick_probe(

            train_design_features[
                feature_name
            ],

            val_design_features[
                feature_name
            ],

            "DESIGN_" + feature_name
        )
    )


diagnostic_results.append(

    quick_probe(

        train_interaction_features[
            "domain_process_counts"
        ],

        val_interaction_features[
            "domain_process_counts"
        ],

        "INTERACTION_domain_process_counts"
    )
)


diagnostic_df = pd.DataFrame(
    diagnostic_results
)


print(
    diagnostic_df.to_string(
        index=False
    )
)


# Are the 12-cell DESIGN signatures themselves unique?

booklet_signatures = []


for booklet_id in sorted(
    np.unique(
        y_train
    )
):

    rows = np.where(
        y_train
        == booklet_id
    )[0]


    signatures = np.unique(

        train_design_features[
            "domain_process_counts"
        ][
            rows
        ],

        axis=0
    )


    assert (
        signatures.shape[0]
        == 1
    )


    booklet_signatures.append(
        signatures[0]
    )


booklet_signatures = np.vstack(
    booklet_signatures
)


n_unique_signatures = (
    np.unique(
        booklet_signatures,
        axis=0
    )
    .shape[0]
)


print(
    "\nUnique booklet signatures from "
    "12 domain×process counts:",
    f"{n_unique_signatures}/30"
)

print(
    "Random 30-class BA:",
    round(
        1 / 30,
        4
    )
)

                      feature_set  n_features  accuracy  balanced_accuracy  macro_f1
            DESIGN_process_counts           3  0.286267           0.233333  0.095021
             DESIGN_domain_counts           4  0.306506           0.333333  0.121731
     DESIGN_domain_process_counts          12  0.385746           0.333333  0.148842
INTERACTION_domain_process_counts          12  0.411857           0.320900  0.226052

Unique booklet signatures from 12 domain×process counts: 10/30
Random 30-class BA: 0.0333


In [ ]:
# Construct assessment-design signature labels
# No model training.

all_design_features = build_exposure_features(
    fold0["train"]["design_mask"]
)

train_signature_matrix = (
    all_design_features[
        "domain_process_counts"
    ]
)


# Unique signatures in the training systems

unique_signatures, train_signature_ids = np.unique(
    train_signature_matrix,
    axis=0,
    return_inverse=True
)


print(
    "Unique assessment-design signatures:",
    unique_signatures.shape[0]
)


# Map each 30-class booklet to one signature

booklet_to_signature = {}


for booklet_id in sorted(
    np.unique(
        fold0["train"][
            "booklet_nuisance_id"
        ]
    )
):

    rows = np.where(
        fold0["train"][
            "booklet_nuisance_id"
        ]
        == booklet_id
    )[0]


    signature_ids = np.unique(
        train_signature_ids[
            rows
        ]
    )


    assert (
        len(signature_ids)
        == 1
    )


    booklet_to_signature[
        int(booklet_id)
    ] = int(
        signature_ids[0]
    )


print(
    "Booklet → signature mapping:"
)

for booklet_id in sorted(
    booklet_to_signature
):

    print(
        f"{booklet_id:2d} -> "
        f"{booklet_to_signature[booklet_id]}"
    )


# Add signature IDs to all Fold-0 splits

for split_name in [
    "train",
    "validation",
    "test"
]:

    booklet_ids = (
        fold0[
            split_name
        ][
            "booklet_nuisance_id"
        ]
    )


    fold0[
        split_name
    ][
        "design_signature_id"
    ] = np.array(
        [
            booklet_to_signature[
                int(b)
            ]
            for b
            in booklet_ids
        ],
        dtype=np.int64
    )


# Audit

for split_name in [
    "train",
    "validation",
    "test"
]:

    ids = (
        fold0[
            split_name
        ][
            "design_signature_id"
        ]
    )

    print(
        f"{split_name:10s}",
        "n signatures =",
        len(
            np.unique(
                ids
            )
        ),
        "| range =",
        (
            ids.min(),
            ids.max()
        )
    )

Unique assessment-design signatures: 10
Booklet → signature mapping:
 0 -> 4
 1 -> 5
 2 -> 9
 3 -> 3
 4 -> 1
 5 -> 7
 6 -> 2
 7 -> 8
 8 -> 0
 9 -> 6
10 -> 4
11 -> 5
12 -> 9
13 -> 3
14 -> 1
15 -> 7
16 -> 2
17 -> 8
18 -> 0
19 -> 6
20 -> 4
21 -> 5
22 -> 9
23 -> 3
24 -> 1
25 -> 7
26 -> 2
27 -> 8
28 -> 0
29 -> 6
train      n signatures = 10 | range = (0, 9)
validation n signatures = 10 | range = (0, 9)
test       n signatures = 10 | range = (0, 9)


In [ ]:
# CReST V6: design-robust general creativity branch

BOOKLET_TO_SIGNATURE = np.array(
    [
        booklet_to_signature[i]
        for i in range(30)
    ],
    dtype=np.int64
)

BOOKLET_TO_SIGNATURE_TENSOR = torch.as_tensor(
    BOOKLET_TO_SIGNATURE,
    dtype=torch.long,
    device=device
)


class CReSTV6(CReSTV5):

    def __init__(
        self,
        *args,
        n_design_signatures=10,
        n_heads=8,
        dropout=0.1,
        **kwargs
    ):

        super().__init__(
            *args,
            n_heads=n_heads,
            dropout=dropout,
            **kwargs
        )

        # Domain-blind item fusion

        self.general_item_fusion = nn.Sequential(

            nn.Linear(
                self.d_model,
                self.d_model
            ),

            nn.GELU(),

            nn.LayerNorm(
                self.d_model
            )
        )


        # Transformer over exactly 3 process summaries

        process_layer = nn.TransformerEncoderLayer(

            d_model=self.d_model,
            nhead=n_heads,

            dim_feedforward=(
                2 * self.d_model
            ),

            dropout=dropout,

            activation="gelu",

            batch_first=True,

            norm_first=True
        )


        self.general_process_encoder = (
            nn.TransformerEncoder(

                process_layer,

                num_layers=1,

                norm=nn.LayerNorm(
                    self.d_model
                )
            )
        )


        # Explicit assessment-design adversary

        self.design_signature_classifier = (
            nn.Sequential(

                nn.Linear(
                    self.latent_dim,
                    self.latent_dim
                ),

                nn.GELU(),

                nn.Linear(
                    self.latent_dim,
                    n_design_signatures
                )
            )
        )


    def build_design_robust_general(
        self,
        batch
    ):

        credit = batch[
            "credit_tokens"
        ]

        behaviour = batch[
            "behaviour_tokens"
        ]

        process_values = batch[
            "process_values"
        ]

        process_mask = batch[
            "process_mask"
        ]

        interaction_mask = batch[
            "interaction_mask"
        ]


        batch_size = (
            credit.shape[0]
        )


        # Process identity only:
        # NO domain embedding
        # NO item embedding

        process_meta = (
            self.ideation_embedding(
                self.item_ideation_ids
            )
            .unsqueeze(0)
            .expand(
                batch_size,
                -1,
                -1
            )
        )


        process_input = torch.cat(
            [
                process_values,
                process_mask.float()
            ],
            dim=-1
        )


        general_items = (

            self.credit_embedding(
                credit
            )

            +

            self.behaviour_embedding(
                behaviour
            )

            +

            self.process_projection(
                process_input
            )

            +

            process_meta
        )


        general_items = (
            self.general_item_fusion(
                general_items
            )
        )


        # PRE-TRANSFORMER mean pooling by ideation process
        #
        # Exactly 3 summaries:
        # GDI / GCI / EII

        pooled = []
        available = []


        for process_id in range(
            self.n_processes
        ):

            selector = (

                self.item_ideation_ids
                .unsqueeze(0)

                == process_id
            )


            valid = (

                interaction_mask
                &
                selector
            )


            is_available = (
                valid.any(
                    dim=1
                )
            )


            summary = masked_mean(

                general_items,
                valid,
                dim=1
            )


            summary = torch.where(

                is_available.unsqueeze(-1),

                summary,

                torch.zeros_like(
                    summary
                )
            )


            pooled.append(
                summary
            )

            available.append(
                is_available
            )


        pooled = torch.stack(
            pooled,
            dim=1
        )


        available = torch.stack(
            available,
            dim=1
        )


        # Add only process identity to the 3 summaries

        process_ids = torch.arange(
            self.n_processes,
            device=credit.device
        )


        pooled = (

            pooled

            +

            self.ideation_embedding(
                process_ids
            )
            .unsqueeze(0)
        )


        # Safety
        safe_available = (
            available.clone()
        )


        no_available = (
            ~safe_available.any(
                dim=1
            )
        )


        if no_available.any():

            safe_available[
                no_available,
                0
            ] = True


        process_encoded = (
            self.general_process_encoder(

                pooled,

                src_key_padding_mask=(
                    ~safe_available
                )
            )
        )


        # Equal available-process pooling

        weights = (
            available
            .float()
            .unsqueeze(-1)
        )


        balanced = (

            (
                process_encoded
                * weights
            )
            .sum(
                dim=1
            )

            /

            weights.sum(
                dim=1
            )
            .clamp_min(
                1.0
            )
        )


        balanced = (
            self.process_balance_norm(
                balanced
            )
        )


        z_general = (
            self.creative_projection(
                balanced
            )
        )


        return (
            z_general,
            process_encoded,
            available
        )


    def forward(
        self,
        batch,
        lambda_booklet_grl=1.0,
        lambda_country_grl=1.0,
        lambda_design_grl=1.0
    ):

        # Existing V5 machinery remains for:
        # item reconstruction, domain-specific branch,
        # context branch, etc.

        out = super().forward(

            batch,

            lambda_booklet_grl=0.0,

            lambda_country_grl=0.0
        )


        # Replace V5 z_general

        (
            z_general,
            process_summary_tokens,
            process_summary_available
        ) = self.build_design_robust_general(
            batch
        )


        out[
            "z_general"
        ] = z_general

        out[
            "z_creative"
        ] = z_general

        out[
            "ideation_process_pooled"
        ] = process_summary_tokens

        out[
            "ideation_process_available"
        ] = process_summary_available


        # Rebuild general + domain-specific representations

        z_domain_creative = (

            z_general.unsqueeze(1)

            +

            out[
                "z_domain_specific"
            ]
        )


        z_domain_creative = torch.where(

            out[
                "domain_available"
            ]
            .unsqueeze(-1),

            z_domain_creative,

            torch.zeros_like(
                z_domain_creative
            )
        )


        out[
            "z_domain_creative"
        ] = (
            z_domain_creative
        )


        # Recompute decomposition decoder

        domain_latent_per_item = (
            z_domain_creative.index_select(

                dim=1,

                index=self.item_domain_ids
            )
        )


        decoder_meta = (
            self.decoder_metadata(

                batch[
                    "credit_tokens"
                ].shape[0],

                batch[
                    "credit_tokens"
                ].device
            )
        )


        decomposition_tokens = (

            self.domain_latent_decoder(
                domain_latent_per_item
            )

            +

            decoder_meta
        )


        out[
            "decomposition_credit_logits"
        ] = (

            self.decomposition_credit_head(
                decomposition_tokens
            )
        )


        # Recompute nuisance heads from NEW z_general

        out[
            "booklet_logits"
        ] = (

            self.booklet_classifier(

                grad_reverse(
                    z_general,
                    lambda_booklet_grl
                )
            )
        )


        out[
            "creative_country_logits"
        ] = (

            self.creative_country_classifier(

                grad_reverse(
                    z_general,
                    lambda_country_grl
                )
            )
        )


        out[
            "design_signature_logits"
        ] = (

            self.design_signature_classifier(

                grad_reverse(
                    z_general,
                    lambda_design_grl
                )
            )
        )


        return out


print(
    "CReST V6 architecture ready."
)

CReST V6 architecture ready.


In [ ]:
# Build and smoke-test CReST V6

V6_DESIGN_SIGNATURE_WEIGHT = 0.10


def build_crest_v6_model(
    seed=GLOBAL_SEED
):

    set_global_seed(
        seed
    )


    m = CReSTV6(

        item_domain_ids=item_domain_ids,

        item_ideation_ids=item_ideation_ids,

        student_n_continuous=len(
            fold0[
                "student_preprocessor"
            ].continuous_vars
        ),

        student_cat_cardinalities=(
            student_model_cardinalities
        ),

        school_n_continuous=len(
            fold0[
                "school_preprocessor"
            ].continuous_vars
        ),

        school_cat_cardinalities=(
            school_model_cardinalities
        ),

        n_train_countries=len(
            fold0[
                "country_to_train_id"
            ]
        ),

        n_booklets=30,

        n_design_signatures=10,

        n_domains=4,

        n_processes=3,

        d_model=int(
            CONFIG["d_model"]
        ),

        latent_dim=int(
            CONFIG["latent_dim"]
        ),

        n_heads=int(
            CONFIG["n_heads"]
        ),

        n_layers=int(
            CONFIG["n_layers"]
        ),

        dropout=float(
            CONFIG["dropout"]
        )
    )


    # Context reconstruction heads
    #
    # Encoder uses one extra MASK token per categorical
    # feature, but reconstruction predicts only the original
    # base categories.

    target_cardinalities = (

        student_base_cardinalities

        +

        school_base_cardinalities
    )


    m.context_categorical_heads = (
        nn.ModuleList(
            [
                nn.Linear(
                    int(
                        CONFIG[
                            "latent_dim"
                        ]
                    ),
                    cardinality
                )

                for cardinality
                in target_cardinalities
            ]
        )
    )


    return m.to(
        device
    )


# V6 objective

def compute_crest_v6_objective(
    model,
    batch,
    lambda_booklet_grl=1.0,
    lambda_country_grl=1.0,
    lambda_design_grl=1.0
):

    # Standard CReST losses
    #
    # Since model is CReSTV6, compute_crest_v5_objective()
    # automatically calls the V6 forward method and therefore
    # uses the new design-robust z_general.

    total_base, losses, output = (
        compute_crest_v5_objective(

            model,

            batch,

            lambda_booklet_grl=(
                lambda_booklet_grl
            ),

            lambda_country_grl=(
                lambda_country_grl
            )
        )
    )


    # Assessment-design signature target
    #
    # 30 booklets -> 10 structural design signatures.

    signature_targets = (
        BOOKLET_TO_SIGNATURE_TENSOR[
            batch[
                "booklet_nuisance_id"
            ]
        ]
    )


    # Design-signature adversarial classification loss

    loss_design_signature = (
        F.cross_entropy(

            output[
                "design_signature_logits"
            ],

            signature_targets
        )
    )


    losses[
        "design_signature"
    ] = (
        loss_design_signature
    )


    # Total V6 objective

    total = (

        total_base

        +

        V6_DESIGN_SIGNATURE_WEIGHT
        *
        loss_design_signature
    )


    return (
        total,
        losses,
        output
    )


# Instantiate V6

model_v6 = build_crest_v6_model(
    seed=GLOBAL_SEED
)


print(
    "V6 trainable parameters:",
    sum(
        p.numel()
        for p in model_v6.parameters()
        if p.requires_grad
    )
)


# Small smoke-test batch

smoke_batch = next(
    iter(
        train_loader
    )
)


SMOKE_N = min(
    64,
    smoke_batch[
        "credit_tokens"
    ].shape[0]
)


smoke_batch = {

    k:
        v[:SMOKE_N]
        .to(device)

    for k, v
    in smoke_batch.items()
}


# Forward + backward

set_global_seed(
    GLOBAL_SEED + 20
)


model_v6.train()


model_v6.zero_grad(
    set_to_none=True
)


total_v6, losses_v6, output_v6 = (
    compute_crest_v6_objective(

        model_v6,

        smoke_batch,

        lambda_booklet_grl=0.5,

        lambda_country_grl=0.5,

        lambda_design_grl=0.5
    )
)


assert torch.isfinite(
    total_v6
), "V6 total loss is not finite."


total_v6.backward()


# Gradient audit

critical_v6_gradients = {

    "general_item_fusion":
        grad_norm(
            model_v6
            .general_item_fusion[0]
            .weight
        ),

    "general_process_encoder":
        grad_norm(
            model_v6
            .general_process_encoder
            .layers[0]
            .self_attn
            .in_proj_weight
        ),

    "creative_projection":
        grad_norm(
            model_v6
            .creative_projection[0]
            .weight
        ),

    "design_signature_classifier":
        grad_norm(
            model_v6
            .design_signature_classifier[0]
            .weight
        )
}


# Report

print(
    "\nV6 total loss:",
    round(
        float(
            total_v6.detach().cpu()
        ),
        6
    )
)


print(
    "\nz_general:",
    tuple(
        output_v6[
            "z_general"
        ].shape
    )
)


print(
    "Process summaries:",
    tuple(
        output_v6[
            "ideation_process_pooled"
        ].shape
    )
)


print(
    "Process availability:",
    tuple(
        output_v6[
            "ideation_process_available"
        ].shape
    )
)


print(
    "Design logits:",
    tuple(
        output_v6[
            "design_signature_logits"
        ].shape
    )
)


print(
    "\nDesign signature CE:",
    round(
        float(
            losses_v6[
                "design_signature"
            ].detach().cpu()
        ),
        6
    )
)


print(
    "\nCritical V6 gradient norms:"
)


for name, value in (
    critical_v6_gradients.items()
):

    print(
        f"{name:30s}",
        f"{value:.6f}"
    )


assert all(
    np.isfinite(v)
    and v > 0

    for v
    in critical_v6_gradients.values()
)


print(
    "\nCReST V6 backward smoke test: PASS"
)

V6 trainable parameters: 1592886

V6 total loss: 4.108968

z_general: (64, 64)
Process summaries: (64, 3, 128)
Process availability: (64, 3)
Design logits: (64, 10)

Design signature CE: 2.300082

Critical V6 gradient norms:
general_item_fusion            0.340467
general_process_encoder        0.093264
creative_projection            0.369028
design_signature_classifier    0.029576

CReST V6 backward smoke test: PASS


In [ ]:
# Finalise V6 GRL scheduling before pilot training

class CReSTV6Final(CReSTV6):

    def forward(
        self,
        batch,
        lambda_booklet_grl=1.0,
        lambda_country_grl=1.0,
        lambda_design_grl=None
    ):

        # If design-specific GRL is not explicitly supplied,
        # it follows the booklet/design nuisance schedule.
        if lambda_design_grl is None:
            lambda_design_grl = lambda_booklet_grl

        return super().forward(
            batch,
            lambda_booklet_grl=lambda_booklet_grl,
            lambda_country_grl=lambda_country_grl,
            lambda_design_grl=lambda_design_grl
        )


def build_crest_v6_model(
    seed=GLOBAL_SEED
):

    set_global_seed(seed)

    m = CReSTV6Final(

        item_domain_ids=item_domain_ids,
        item_ideation_ids=item_ideation_ids,

        student_n_continuous=len(
            fold0["student_preprocessor"].continuous_vars
        ),

        student_cat_cardinalities=(
            student_model_cardinalities
        ),

        school_n_continuous=len(
            fold0["school_preprocessor"].continuous_vars
        ),

        school_cat_cardinalities=(
            school_model_cardinalities
        ),

        n_train_countries=len(
            fold0["country_to_train_id"]
        ),

        n_booklets=30,
        n_design_signatures=10,
        n_domains=4,
        n_processes=3,

        d_model=int(CONFIG["d_model"]),
        latent_dim=int(CONFIG["latent_dim"]),
        n_heads=int(CONFIG["n_heads"]),
        n_layers=int(CONFIG["n_layers"]),
        dropout=float(CONFIG["dropout"])
    )


    # Decoder predicts base categories only.
    target_cardinalities = (
        student_base_cardinalities
        +
        school_base_cardinalities
    )

    m.context_categorical_heads = nn.ModuleList(
        [
            nn.Linear(
                int(CONFIG["latent_dim"]),
                cardinality
            )
            for cardinality
            in target_cardinalities
        ]
    )

    return m.to(device)


# V6 objective

def compute_crest_v6_objective(
    model,
    batch,
    lambda_nuisance_grl=1.0,
    lambda_country_grl=1.0
):

    # Base objective.
    # Because the model is CReSTV6Final, design GRL
    # automatically follows lambda_nuisance_grl.
    total_base, losses, output = (
        compute_crest_v5_objective(

            model,
            batch,

            lambda_booklet_grl=(
                lambda_nuisance_grl
            ),

            lambda_country_grl=(
                lambda_country_grl
            )
        )
    )


    # 30 booklets -> 10 structural design signatures
    signature_targets = (
        BOOKLET_TO_SIGNATURE_TENSOR[
            batch["booklet_nuisance_id"]
        ]
    )


    loss_design_signature = F.cross_entropy(
        output["design_signature_logits"],
        signature_targets
    )


    losses["design_signature"] = (
        loss_design_signature
    )


    total = (
        total_base
        +
        V6_DESIGN_SIGNATURE_WEIGHT
        *
        loss_design_signature
    )


    return (
        total,
        losses,
        output
    )


model_v6 = build_crest_v6_model()


print(
    "Final V6 trainable parameters:",
    sum(
        p.numel()
        for p in model_v6.parameters()
        if p.requires_grad
    )
)

print(
    "V6 GRL scheduling corrected."
)

Final V6 trainable parameters: 1592886
V6 GRL scheduling corrected.


In [ ]:
# CReST V6 Fold-0 pilot training

V6_PILOT_EPOCHS = 5


# Fresh reproducible country-balanced loader

v6_sampler = WeightedRandomSampler(

    weights=torch.as_tensor(
        sampling_weights,
        dtype=torch.double
    ),

    num_samples=len(
        train_dataset
    ),

    replacement=True,

    generator=(
        torch.Generator()
        .manual_seed(
            GLOBAL_SEED + 600
        )
    )
)


v6_train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    sampler=v6_sampler,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


# Model / optimiser

model_v6 = build_crest_v6_model(
    seed=GLOBAL_SEED
)


optimizer_v6 = torch.optim.AdamW(

    model_v6.parameters(),

    lr=1e-4,

    weight_decay=1e-4
)


scheduler_v6 = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(

        optimizer_v6,

        mode="min",

        factor=0.5,

        patience=2,

        min_lr=1e-6
    )
)


# Checkpoint

v6_checkpoint = (
    CHECKPOINT_FOLD0_DIR
    / "crest_v6_pilot_best.pt"
)


v6_history = []

best_v6_val = np.inf
best_v6_epoch = None


# Training

for epoch in range(
    1,
    V6_PILOT_EPOCHS + 1
):

    model_v6.train()


    running_total = 0.0
    running_recon = 0.0

    running_booklet = 0.0
    running_design = 0.0

    running_country_context = 0.0
    running_country_creative = 0.0

    n_seen = 0


    for batch_idx, batch in enumerate(
        v6_train_loader
    ):

        batch = {
            k: v.to(device)
            for k, v
            in batch.items()
        }


        # GRL schedule

        progress = (

            (
                epoch - 1
            )

            +

            batch_idx
            /
            max(
                len(v6_train_loader),
                1
            )

        ) / V6_PILOT_EPOCHS


        lambda_grl = float(
            grl_schedule(
                progress
            )
        )


        # Optimisation

        optimizer_v6.zero_grad(
            set_to_none=True
        )


        total_loss, losses, _ = (
            compute_crest_v6_objective(

                model_v6,

                batch,

                lambda_nuisance_grl=(
                    lambda_grl
                ),

                lambda_country_grl=(
                    lambda_grl
                )
            )
        )


        total_loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model_v6.parameters(),
            max_norm=1.0
        )


        optimizer_v6.step()


        # Training summaries

        n_batch = (
            batch[
                "credit_tokens"
            ].shape[0]
        )


        recon = (
            reconstruction_score_from_losses(
                losses
            )
        )


        running_total += (
            float(
                total_loss.detach().cpu()
            )
            *
            n_batch
        )


        running_recon += (
            float(
                recon.detach().cpu()
            )
            *
            n_batch
        )


        running_booklet += (
            float(
                losses[
                    "booklet"
                ].detach().cpu()
            )
            *
            n_batch
        )


        running_design += (
            float(
                losses[
                    "design_signature"
                ].detach().cpu()
            )
            *
            n_batch
        )


        running_country_context += (
            float(
                losses[
                    "country_context"
                ].detach().cpu()
            )
            *
            n_batch
        )


        running_country_creative += (
            float(
                losses[
                    "country_creative"
                ].detach().cpu()
            )
            *
            n_batch
        )


        n_seen += (
            n_batch
        )


    # Validation on unseen systems

    val_score, val_country_scores = (
        evaluate_country_equal_reconstruction(
            model_v6
        )
    )


    scheduler_v6.step(
        val_score
    )


    lr_now = (
        optimizer_v6
        .param_groups[0]["lr"]
    )


    epoch_row = {

        "epoch":
            epoch,

        "train_total":
            running_total
            / n_seen,

        "train_reconstruction":
            running_recon
            / n_seen,

        "train_booklet_ce":
            running_booklet
            / n_seen,

        "train_design_signature_ce":
            running_design
            / n_seen,

        "train_country_context_ce":
            running_country_context
            / n_seen,

        "train_country_creative_ce":
            running_country_creative
            / n_seen,

        "val_country_equal_reconstruction":
            val_score,

        "learning_rate":
            lr_now
    }


    v6_history.append(
        epoch_row
    )


    # Best checkpoint

    if val_score < best_v6_val:

        best_v6_val = (
            val_score
        )

        best_v6_epoch = (
            epoch
        )


        torch.save(

            {
                "epoch":
                    epoch,

                "model_state_dict":
                    model_v6.state_dict(),

                "optimizer_state_dict":
                    optimizer_v6.state_dict(),

                "val_country_equal_reconstruction":
                    val_score,

                "val_country_scores":
                    val_country_scores
            },

            v6_checkpoint
        )


    # Epoch report

    print(

        f"Epoch {epoch:02d} | "

        f"train={epoch_row['train_total']:.4f} | "

        f"recon={epoch_row['train_reconstruction']:.4f} | "

        f"val={val_score:.4f} | "

        f"bookletCE={epoch_row['train_booklet_ce']:.4f} | "

        f"designCE={epoch_row['train_design_signature_ce']:.4f} | "

        f"ctxCountryCE={epoch_row['train_country_context_ce']:.4f} | "

        f"creativeCountryCE={epoch_row['train_country_creative_ce']:.4f} | "

        f"GRL={lambda_grl:.3f} | "

        f"lr={lr_now:.1e}"
    )


# Save history

v6_history_df = pd.DataFrame(
    v6_history
)


v6_history_df.to_csv(

    RESULTS_DIR
    / "fold0_v6_pilot_history.csv",

    index=False
)


print(
    "\nBest V6 pilot epoch:",
    best_v6_epoch
)


print(
    "Best country-equal validation reconstruction:",
    round(
        best_v6_val,
        6
    )
)


print(
    "Checkpoint:",
    v6_checkpoint
)

Epoch 01 | train=3.0140 | recon=1.6688 | val=1.4933 | bookletCE=3.3411 | designCE=2.2995 | ctxCountryCE=3.7034 | creativeCountryCE=3.7771 | GRL=0.761 | lr=1.0e-04
Epoch 02 | train=2.7898 | recon=1.5011 | val=1.4570 | bookletCE=3.2378 | designCE=2.3091 | ctxCountryCE=3.2528 | creativeCountryCE=3.7973 | GRL=0.964 | lr=1.0e-04
Epoch 03 | train=2.7249 | recon=1.4721 | val=1.4358 | bookletCE=3.1028 | designCE=2.3016 | ctxCountryCE=2.9661 | creativeCountryCE=3.8197 | GRL=0.995 | lr=1.0e-04
Epoch 04 | train=2.6817 | recon=1.4599 | val=1.4304 | bookletCE=2.9318 | designCE=2.3069 | ctxCountryCE=2.8257 | creativeCountryCE=3.8399 | GRL=0.999 | lr=1.0e-04
Epoch 05 | train=2.6443 | recon=1.4506 | val=1.4241 | bookletCE=2.8281 | designCE=2.3077 | ctxCountryCE=2.7630 | creativeCountryCE=3.8132 | GRL=1.000 | lr=1.0e-04

Best V6 pilot epoch: 5
Best country-equal validation reconstruction: 1.424053
Checkpoint: PROJECT_ROOT\checkpoints\fold_0\crest_v6_pilot_best.pt


In [ ]:
# Restore best V6 checkpoint and extract frozen latents

model_v6 = build_crest_v6_model(
    seed=GLOBAL_SEED
)


checkpoint_v6 = torch.load(
    v6_checkpoint,
    map_location=device
)


model_v6.load_state_dict(
    checkpoint_v6["model_state_dict"]
)

model_v6.eval()


print(
    "Restored V6 pilot epoch:",
    checkpoint_v6["epoch"]
)

print(
    "Validation reconstruction:",
    round(
        checkpoint_v6[
            "val_country_equal_reconstruction"
        ],
        6
    )
)


# Deterministic loaders

probe_train_loader = DataLoader(
    train_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=0
)

probe_validation_loader = DataLoader(
    validation_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=0
)


# Frozen representation extraction

@torch.no_grad()
def extract_v6_latents(
    model,
    loader
):

    general_list = []
    context_list = []
    booklet_list = []
    design_list = []


    for batch in loader:

        batch = {
            k: v.to(device)
            for k, v
            in batch.items()
        }


        out = model(

            batch,

            lambda_booklet_grl=0.0,

            lambda_country_grl=0.0,

            lambda_design_grl=0.0
        )


        booklet = (
            batch[
                "booklet_nuisance_id"
            ]
        )


        design_signature = (
            BOOKLET_TO_SIGNATURE_TENSOR[
                booklet
            ]
        )


        general_list.append(
            out["z_general"]
            .cpu()
            .numpy()
        )

        context_list.append(
            out["z_context"]
            .cpu()
            .numpy()
        )

        booklet_list.append(
            booklet
            .cpu()
            .numpy()
        )

        design_list.append(
            design_signature
            .cpu()
            .numpy()
        )


    return {

        "z_general":
            np.concatenate(
                general_list,
                axis=0
            ),

        "z_context":
            np.concatenate(
                context_list,
                axis=0
            ),

        "booklet":
            np.concatenate(
                booklet_list,
                axis=0
            ),

        "design_signature":
            np.concatenate(
                design_list,
                axis=0
            )
    }


v6_train_latents = (
    extract_v6_latents(
        model_v6,
        probe_train_loader
    )
)


v6_validation_latents = (
    extract_v6_latents(
        model_v6,
        probe_validation_loader
    )
)


print(
    "\nTrain z_general:",
    v6_train_latents[
        "z_general"
    ].shape
)

print(
    "Validation z_general:",
    v6_validation_latents[
        "z_general"
    ].shape
)

print(
    "Booklet classes:",
    len(
        np.unique(
            v6_validation_latents[
                "booklet"
            ]
        )
    )
)

print(
    "Design-signature classes:",
    len(
        np.unique(
            v6_validation_latents[
                "design_signature"
            ]
        )
    )
)

Restored V6 pilot epoch: 5
Validation reconstruction: 1.424053

Train z_general: (99966, 64)
Validation z_general: (20949, 64)
Booklet classes: 30
Design-signature classes: 10


In [ ]:
# Frozen cross-system nuisance probes for V6

def frozen_probe(
    X_train,
    y_train,
    X_val,
    y_val,
    latent_name,
    target_name
):

    probe = make_pipeline(

        StandardScaler(),

        LogisticRegression(
            max_iter=500,
            solver="lbfgs",
            class_weight="balanced",
            random_state=GLOBAL_SEED
        )
    )


    probe.fit(
        X_train,
        y_train
    )


    pred = probe.predict(
        X_val
    )


    return {

        "latent":
            latent_name,

        "target":
            target_name,

        "accuracy":
            accuracy_score(
                y_val,
                pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_val,
                pred
            ),

        "macro_f1":
            f1_score(
                y_val,
                pred,
                average="macro"
            ),

        "chance_balanced_accuracy":
            1.0
            /
            len(
                np.unique(
                    y_val
                )
            )
    }


v6_probe_results = []


for latent_name in [
    "z_general",
    "z_context"
]:

    # 30-class booklet

    v6_probe_results.append(

        frozen_probe(

            v6_train_latents[
                latent_name
            ],

            v6_train_latents[
                "booklet"
            ],

            v6_validation_latents[
                latent_name
            ],

            v6_validation_latents[
                "booklet"
            ],

            latent_name,

            "booklet_30"
        )
    )


    # 10-class design signature

    v6_probe_results.append(

        frozen_probe(

            v6_train_latents[
                latent_name
            ],

            v6_train_latents[
                "design_signature"
            ],

            v6_validation_latents[
                latent_name
            ],

            v6_validation_latents[
                "design_signature"
            ],

            latent_name,

            "design_signature_10"
        )
    )


v6_probe_df = pd.DataFrame(
    v6_probe_results
)


v6_probe_df.to_csv(

    RESULTS_DIR
    / "fold0_v6_pilot_nuisance_probes.csv",

    index=False
)


print(
    v6_probe_df.to_string(
        index=False
    )
)


print(
    "\nReferences:"
)

print(
    "V3 z_general booklet BA : 0.3389"
)

print(
    "V5 z_general booklet BA : 0.3302"
)

print(
    "Booklet chance BA       : 0.0333"
)

print(
    "Design-signature chance : 0.1000"
)

   latent              target  accuracy  balanced_accuracy  macro_f1  chance_balanced_accuracy
z_general          booklet_30  0.113848           0.126755  0.083900                  0.033333
z_general design_signature_10  0.375197           0.375590  0.372083                  0.100000
z_context          booklet_30  0.007351           0.035274  0.008441                  0.033333
z_context design_signature_10  0.100482           0.100612  0.097690                  0.100000

References:
V3 z_general booklet BA : 0.3389
V5 z_general booklet BA : 0.3302
Booklet chance BA       : 0.0333
Design-signature chance : 0.1000


In [ ]:
# Diagnose which V6 input channel carries design-signature information
# No neural-network training.

def safe_mean(values, mask, axis=1):
    numerator = np.where(
        mask,
        values,
        0.0
    ).sum(axis=axis)

    denominator = mask.sum(
        axis=axis
    )

    return np.divide(
        numerator,
        denominator,
        out=np.zeros_like(
            numerator,
            dtype=np.float32
        ),
        where=denominator > 0
    )


def build_v6_signal_features(
    split_data
):

    credits = split_data[
        "credit_tokens"
    ]

    perf_mask = split_data[
        "performance_mask"
    ]

    behaviour = split_data[
        "behaviour_tokens"
    ]

    interaction = split_data[
        "interaction_mask"
    ]

    proc_values = split_data[
        "process_values"
    ]

    proc_mask = split_data[
        "process_mask"
    ]


    credit_features = []
    behaviour_features = []
    process_features = []


    for p in range(3):

        item_sel = (
            process_ids_np == p
        )


        # CREDIT features

        c = credits[:, item_sel]

        cm = perf_mask[:, item_sel]


        # no=0, partial=.5, full=1
        score = (
            c.astype(np.float32)
            - 1.0
        ) / 2.0


        mean_credit = safe_mean(
            score,
            cm
        )


        denom_credit = (
            cm.sum(axis=1)
        )


        def credit_rate(
            token
        ):

            num = (
                (
                    (c == token)
                    &
                    cm
                )
                .sum(axis=1)
            )

            return np.divide(
                num,
                denom_credit,
                out=np.zeros(
                    len(c),
                    dtype=np.float32
                ),
                where=denom_credit > 0
            )


        no_rate = credit_rate(1)
        partial_rate = credit_rate(2)
        full_rate = credit_rate(3)


        credit_features.extend(
            [
                mean_credit,
                no_rate,
                partial_rate,
                full_rate
            ]
        )


        # BEHAVIOUR features

        b = behaviour[:, item_sel]

        im = interaction[:, item_sel]


        denom_behaviour = (
            im.sum(axis=1)
        )


        def behaviour_rate(
            token
        ):

            num = (
                (
                    (b == token)
                    &
                    im
                )
                .sum(axis=1)
            )

            return np.divide(
                num,
                denom_behaviour,
                out=np.zeros(
                    len(b),
                    dtype=np.float32
                ),
                where=denom_behaviour > 0
            )


        # 1=scorable
        # 2=no response
        # 3=not reached
        behaviour_features.extend(
            [
                behaviour_rate(1),
                behaviour_rate(2),
                behaviour_rate(3)
            ]
        )


        # PROCESS features

        pv = proc_values[
            :,
            item_sel,
            :
        ]

        pm = proc_mask[
            :,
            item_sel,
            :
        ]


        # Three scaled process channels
        for channel in range(3):

            values = pv[
                :,
                :,
                channel
            ]

            mask = pm[
                :,
                :,
                channel
            ]


            mean_value = safe_mean(
                values,
                mask
            )


            observed_rate = (
                mask.mean(
                    axis=1
                )
                .astype(
                    np.float32
                )
            )


            process_features.extend(
                [
                    mean_value,
                    observed_rate
                ]
            )


    return {

        "credit":
            np.column_stack(
                credit_features
            ).astype(
                np.float32
            ),

        "behaviour":
            np.column_stack(
                behaviour_features
            ).astype(
                np.float32
            ),

        "process":
            np.column_stack(
                process_features
            ).astype(
                np.float32
            )
    }


train_signal = build_v6_signal_features(
    fold0["train"]
)

val_signal = build_v6_signal_features(
    fold0["validation"]
)


y_train_signature = (
    fold0["train"][
        "design_signature_id"
    ]
)

y_val_signature = (
    fold0["validation"][
        "design_signature_id"
    ]
)


# Probe helper

def signature_probe(
    X_train,
    X_val,
    name
):

    clf = make_pipeline(

        StandardScaler(),

        LogisticRegression(
            max_iter=500,
            solver="lbfgs",
            class_weight="balanced",
            random_state=GLOBAL_SEED
        )
    )


    clf.fit(
        X_train,
        y_train_signature
    )


    pred = clf.predict(
        X_val
    )


    return {

        "feature_set":
            name,

        "n_features":
            X_train.shape[1],

        "accuracy":
            accuracy_score(
                y_val_signature,
                pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_val_signature,
                pred
            ),

        "macro_f1":
            f1_score(
                y_val_signature,
                pred,
                average="macro"
            )
    }


diagnostic_results = []


# Individual channels
for name in [
    "credit",
    "behaviour",
    "process"
]:

    diagnostic_results.append(

        signature_probe(

            train_signal[name],

            val_signal[name],

            name
        )
    )


# Credit + behaviour
diagnostic_results.append(

    signature_probe(

        np.column_stack(
            [
                train_signal["credit"],
                train_signal["behaviour"]
            ]
        ),

        np.column_stack(
            [
                val_signal["credit"],
                val_signal["behaviour"]
            ]
        ),

        "credit+behaviour"
    )
)


# All V6 signal families
diagnostic_results.append(

    signature_probe(

        np.column_stack(
            [
                train_signal["credit"],
                train_signal["behaviour"],
                train_signal["process"]
            ]
        ),

        np.column_stack(
            [
                val_signal["credit"],
                val_signal["behaviour"],
                val_signal["process"]
            ]
        ),

        "credit+behaviour+process"
    )
)


signal_diagnostic_df = pd.DataFrame(
    diagnostic_results
)


print(
    signal_diagnostic_df.to_string(
        index=False
    )
)


print(
    "\nChance balanced accuracy: 0.1000"
)

print(
    "V6 z_general design-signature BA: 0.3756"
)

             feature_set  n_features  accuracy  balanced_accuracy  macro_f1
                  credit          12  0.196000           0.196594  0.188648
               behaviour           9  0.169268           0.170772  0.151439
                 process          18  0.693351           0.693489  0.692967
        credit+behaviour          21  0.227696           0.228308  0.224581
credit+behaviour+process          39  0.759941           0.760590  0.759713

Chance balanced accuracy: 0.1000
V6 z_general design-signature BA: 0.3756


In [ ]:
# Separate process-value leakage from process-observation leakage
# No neural-network training.

# build_v6_signal_features() created process features in this order:
#
# For each ideation process (3):
#   channel 0 mean, channel 0 observed rate,
#   channel 1 mean, channel 1 observed rate,
#   channel 2 mean, channel 2 observed rate
#
# => 6 features per ideation process, 18 total.


process_value_cols = []
process_observation_cols = []


for p in range(3):

    base = p * 6

    process_value_cols.extend(
        [
            base + 0,
            base + 2,
            base + 4
        ]
    )

    process_observation_cols.extend(
        [
            base + 1,
            base + 3,
            base + 5
        ]
    )


X_train_process = (
    train_signal["process"]
)

X_val_process = (
    val_signal["process"]
)


process_value_train = (
    X_train_process[
        :,
        process_value_cols
    ]
)

process_value_val = (
    X_val_process[
        :,
        process_value_cols
    ]
)


process_observation_train = (
    X_train_process[
        :,
        process_observation_cols
    ]
)

process_observation_val = (
    X_val_process[
        :,
        process_observation_cols
    ]
)


process_component_results = []


# Values only

process_component_results.append(

    signature_probe(

        process_value_train,

        process_value_val,

        "process_values_only"
    )
)


# Observation / mask rates only

process_component_results.append(

    signature_probe(

        process_observation_train,

        process_observation_val,

        "process_observation_only"
    )
)


# Values + observation

process_component_results.append(

    signature_probe(

        X_train_process,

        X_val_process,

        "process_values+observation"
    )
)


process_component_df = pd.DataFrame(
    process_component_results
)


print(
    process_component_df.to_string(
        index=False
    )
)


print(
    "\nChance balanced accuracy: 0.1000"
)

print(
    "Previous full process BA: 0.6935"
)

               feature_set  n_features  accuracy  balanced_accuracy  macro_f1
       process_values_only           9  0.222397           0.222647  0.211902
  process_observation_only           9  0.620889           0.620964  0.605304
process_values+observation          18  0.693351           0.693489  0.692967

Chance balanced accuracy: 0.1000
Previous full process BA: 0.6935


In [ ]:
# Test a mask-free candidate for the V6 general branch
# No neural-network training.

credit_train = train_signal["credit"]
credit_val = val_signal["credit"]

behaviour_train = train_signal["behaviour"]
behaviour_val = val_signal["behaviour"]

process_values_train = process_value_train
process_values_val = process_value_val


maskfree_results = []


# Credit + process values

maskfree_results.append(

    signature_probe(

        np.column_stack(
            [
                credit_train,
                process_values_train
            ]
        ),

        np.column_stack(
            [
                credit_val,
                process_values_val
            ]
        ),

        "credit+process_values"
    )
)


# Behaviour + process values

maskfree_results.append(

    signature_probe(

        np.column_stack(
            [
                behaviour_train,
                process_values_train
            ]
        ),

        np.column_stack(
            [
                behaviour_val,
                process_values_val
            ]
        ),

        "behaviour+process_values"
    )
)


# Credit + behaviour + process values

maskfree_results.append(

    signature_probe(

        np.column_stack(
            [
                credit_train,
                behaviour_train,
                process_values_train
            ]
        ),

        np.column_stack(
            [
                credit_val,
                behaviour_val,
                process_values_val
            ]
        ),

        "credit+behaviour+process_values"
    )
)


maskfree_df = pd.DataFrame(
    maskfree_results
)


print(
    maskfree_df.to_string(
        index=False
    )
)


print(
    "\nReference:"
)

print(
    "Chance BA                         : 0.1000"
)

print(
    "Process observation only BA       : 0.6210"
)

print(
    "Current V6 z_general design BA    : 0.3756"
)

print(
    "Current V6 z_general booklet BA   : 0.1268"
)

                    feature_set  n_features  accuracy  balanced_accuracy  macro_f1
          credit+process_values          21  0.280109           0.280456  0.275438
       behaviour+process_values          18  0.270228           0.270621  0.263017
credit+behaviour+process_values          30  0.317867           0.318307  0.315190

Reference:
Chance BA                         : 0.1000
Process observation only BA       : 0.6210
Current V6 z_general design BA    : 0.3756
Current V6 z_general booklet BA   : 0.1268


## Development Summary

CReST V5 substantially improved reconstruction performance but retained strong booklet-related information in the general creative representation. Analysis showed that the 30 PISA booklets collapse into 10 recurring domain-by-ideation-process assessment-design signatures. CReST V6 therefore introduced a domain-blind, process-balanced general branch and an explicit design-signature adversary. This reduced frozen cross-system booklet balanced accuracy from 0.3302 (V5) to 0.1268, while maintaining strong validation reconstruction performance (1.4241). However, the 10-class design-signature probe remained above chance (BA = 0.3756 vs. 0.10). Diagnostic analyses identified process-observation patterns as the main source of remaining design leakage (BA = 0.6210), followed by weaker residual information in process values, credit, and response behaviour. The next stage therefore focuses on train-only item-conditioned residualization before finalising the design-robust general representation.